# 048 - Train W1, Offline Augmented-Test Evaluation, and TTA

This Kaggle notebook first trains the canonical W1 model because no checkpoint is assumed to exist. It then evaluates:

- **M0:** standard W1 reference on the original grouped test split;
- **M1:** six offline geometric test views, with geometry applied before W1;
- **M2:** single-view custom-evaluator control;
- **M3:** six-view W1 TTA with inverse mapping and mask fusion.

Standard AP uses Ultralytics validation confidence behavior (`conf=0.001`, supplied explicitly for auditability). Operational healthy-negative, disease-miss, and count diagnostics use `conf=0.25`.


## Primary Outputs

- strict W1 training summary and paper row;
- seed-42 grouped split manifest, summary, and fingerprint;
- trained W1 `best.pt`;
- M0/M1 standard Ultralytics metrics, including per-view and pooled rows;
- M2/M3 custom-evaluator metrics and per-class rows;
- healthy-negative, disease-miss, count, and healthy-aware diagnostics;
- preprocessing/inference/fusion timing records;
- compact reports-and-checkpoint ZIP package.


## Run Controls

In [ ]:
# Run controls for ONLY_fourier path 0.
from pathlib import Path

EXECUTION_PLAN = '048_w1_train_aug_test_tta_kaggle_seed42'
SMOKE_RUN = False
SEEDS = [42]
DISABLE_ULTRALYTICS_ALBUMENTATIONS = None  # Controlled per A-H mode via default_hook.
PINNED_ULTRALYTICS_VERSION = '8.4.62'

# Post-training evaluation controls.
RUN_M0_STANDARD_REFERENCE = True
RUN_M1_OFFLINE_AUG_TEST = not SMOKE_RUN
RUN_M2_SINGLE_VIEW_CONTROL = not SMOKE_RUN
RUN_M3_SIX_VIEW_TTA = not SMOKE_RUN
EXPECTED_SPLIT_FINGERPRINT = '1ffd4a250deb11598f09a0a25d1cd1522811b6e010b0228c3c60176029ff9fcb'
STANDARD_AP_CONF = 0.001
DIAGNOSTIC_CONF = 0.25
VAL_BATCH = 16
TTA_INFERENCE_BATCH = 1


YOLO_MODELS = [
    'yolo11n-seg.pt',
]

SPLIT_POLICIES = [
    {
        'key': 'stratified_grouped_specimen',
        'name': 'Stratified grouped-specimen split',
        'description': 'Disease-stratified split where all images from one shrimp stay together.',
    },
]

if Path('/kaggle/working').exists():
    WORK_DIR = Path('/kaggle/working')
elif Path('/content').exists():
    WORK_DIR = Path('/content')
else:
    WORK_DIR = Path.cwd()

DATASET_DIR = WORK_DIR / 'shrimpDisHandSegV2-1'
EXPERIMENT_ROOT = WORK_DIR / 'shrimp_048_w1_train_aug_test_tta_seed42'
RUNS_DIR = WORK_DIR / 'runs' / 'segment'
REPORT_DIR = EXPERIMENT_ROOT / 'reports'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print('EXECUTION_PLAN:', EXECUTION_PLAN)
print('WORK_DIR:', WORK_DIR)
print('DATASET_DIR:', DATASET_DIR)
print('SMOKE_RUN:', SMOKE_RUN)
print('SEEDS:', SEEDS)
print('DISABLE_ULTRALYTICS_ALBUMENTATIONS:', DISABLE_ULTRALYTICS_ALBUMENTATIONS)
print('PINNED_ULTRALYTICS_VERSION:', PINNED_ULTRALYTICS_VERSION)
print('YOLO_MODELS:', YOLO_MODELS)
print('SPLIT_POLICIES:', [p['key'] for p in SPLIT_POLICIES])


## Install Dependencies

In [ ]:
import importlib.metadata
import subprocess
import sys


def installed_version(package_name):
    try:
        return importlib.metadata.version(package_name)
    except importlib.metadata.PackageNotFoundError:
        return None


if installed_version('ultralytics') != PINNED_ULTRALYTICS_VERSION:
    subprocess.check_call([
        sys.executable,
        '-m',
        'pip',
        'install',
        '-q',
        f'ultralytics=={PINNED_ULTRALYTICS_VERSION}',
    ])

for package_name in ['roboflow', 'pandas', 'pyyaml', 'seaborn']:
    if installed_version(package_name) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package_name])

import ultralytics
from ultralytics import YOLO

print('Ultralytics version:', ultralytics.__version__)
print('Expected version:', PINNED_ULTRALYTICS_VERSION)
assert ultralytics.__version__ == PINNED_ULTRALYTICS_VERSION, 'Ultralytics version mismatch.'

## Dataset Download

In [ ]:
import os
from roboflow import Roboflow

ROBOFLOW_API_KEY_DIRECT = 'KOEk0qLzBFDc7zfyxtgs'
ROBOFLOW_WORKSPACE = 'lets-try-this'
ROBOFLOW_PROJECT = 'shrimpdishandsegv2'
ROBOFLOW_VERSION = 1
ROBOFLOW_FORMAT = 'yolo26'


def get_roboflow_api_key():
    if ROBOFLOW_API_KEY_DIRECT.strip():
        return ROBOFLOW_API_KEY_DIRECT.strip()
    try:
        from kaggle_secrets import UserSecretsClient
        key = UserSecretsClient().get_secret('ROBOFLOW_API_KEY')
        if key:
            return key
    except Exception:
        pass
    try:
        from google.colab import userdata
        key = userdata.get('ROBOFLOW_API_KEY')
        if key:
            return key
    except Exception:
        pass
    return os.environ.get('ROBOFLOW_API_KEY', '').strip()


api_key = get_roboflow_api_key()
if not api_key:
    raise RuntimeError(
        'Missing Roboflow API key. Add a Kaggle or Colab Secret named ROBOFLOW_API_KEY, '
        'set the ROBOFLOW_API_KEY environment variable, or temporarily fill ROBOFLOW_API_KEY_DIRECT.'
    )

rf = Roboflow(api_key=api_key)
project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
version = project.version(ROBOFLOW_VERSION)
dataset = version.download(ROBOFLOW_FORMAT, location=str(DATASET_DIR))

base_path = str(DATASET_DIR)
data_yaml_path = str(DATASET_DIR / 'data.yaml')
print('Dataset path:', base_path)
print('data.yaml:', data_yaml_path)


## Leakage-Safe Split Helpers

In [ ]:
import os
import random
import re
import shutil
from collections import defaultdict, Counter
from pathlib import Path

SEED = 42
random.seed(SEED)

base_path = str(DATASET_DIR)
train_path = os.path.join(base_path, 'train')
IMAGE_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')

# Prevent leakage from multiple photos of the same shrimp.
# Expected original filename: <diseasename>-<shrimpid>-img-<imgnum>.jpg
# Roboflow may export names like <diseasename>-<shrimpid>-img-<imgnum>_jpg.rf.<hash>.jpg.
# Example disease names: Healthy, BG, WSSV_BG, WSSV.
GROUP_SPLIT_BY_SHRIMP = True
GROUP_STRATIFY_BY_DISEASE = True
REBUILD_SPLIT_FROM_ALL_SPLITS = True
TRAIN_RATIO = 0.80
VAL_RATIO = 0.10

SHRIMP_NAME_PATTERN = re.compile(
    r'^(?P<disease>Healthy|BG|WSSV_BG|WSSV)-(?P<shrimp_id>.+)-img-(?P<img_num>\d+)$',
    re.IGNORECASE,
)


def normalize_roboflow_stem(stem):
    """Recover the original filename stem from Roboflow-exported names."""
    stem = re.sub(r'_(jpg|jpeg|png|bmp|webp)\.rf\.[0-9a-f]+$', '', stem, flags=re.IGNORECASE)
    stem = re.sub(r'\.rf\.[0-9a-f]+$', '', stem, flags=re.IGNORECASE)
    return stem


for split in ['train', 'valid', 'test']:
    for sub in ['images', 'labels']:
        os.makedirs(os.path.join(base_path, split, sub), exist_ok=True)


def parse_shrimp_group_key(image_name):
    """Return a stable group key so all images from one shrimp stay in one split."""
    stem = normalize_roboflow_stem(Path(image_name).stem)
    match = SHRIMP_NAME_PATTERN.match(stem)
    if not match:
        return f'unparsed::{Path(image_name).stem}', 'unparsed', None, None

    disease = match.group('disease')
    shrimp_id = match.group('shrimp_id')
    img_num = int(match.group('img_num'))
    group_key = f'{disease.lower()}::{shrimp_id}'
    return group_key, disease, shrimp_id, img_num


def image_files_in_split(split):
    image_dir = Path(base_path) / split / 'images'
    return sorted(
        p for p in image_dir.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    )


def move_image_and_label(image_path, target_split):
    target_img_dir = Path(base_path) / target_split / 'images'
    target_lbl_dir = Path(base_path) / target_split / 'labels'
    target_img_dir.mkdir(parents=True, exist_ok=True)
    target_lbl_dir.mkdir(parents=True, exist_ok=True)

    label_name = f'{image_path.stem}.txt'
    label_src = image_path.parent.parent / 'labels' / label_name
    image_dst = target_img_dir / image_path.name
    label_dst = target_lbl_dir / label_name

    if image_path.resolve() != image_dst.resolve():
        if image_dst.exists():
            raise FileExistsError(f'Duplicate image destination would be overwritten: {image_dst}')
        shutil.move(str(image_path), str(image_dst))

    if label_src.exists():
        if label_src.resolve() != label_dst.resolve():
            if label_dst.exists():
                raise FileExistsError(f'Duplicate label destination would be overwritten: {label_dst}')
            shutil.move(str(label_src), str(label_dst))
    else:
        label_dst.write_text('')


def rebuild_train_pool_from_all_splits():
    all_images = []
    for split in ['train', 'valid', 'test']:
        all_images.extend(image_files_in_split(split))

    for image_path in sorted(all_images):
        move_image_and_label(image_path, 'train')

    return image_files_in_split('train')


def remove_yolo_label_caches(root):
    for cache_path in Path(root).glob('**/*.cache'):
        cache_path.unlink()
        print(f'Removed stale cache: {cache_path}')


def disease_for_group(filenames):
    diseases = []
    for filename in filenames:
        _, disease, _, _ = parse_shrimp_group_key(filename)
        diseases.append(disease)
    counts = Counter(diseases)
    if len(counts) > 1:
        print(f'Warning: group has mixed disease names: {dict(counts)}')
    return counts.most_common(1)[0][0]


def split_one_stratum(items):
    n = len(items)
    train_count = int(TRAIN_RATIO * n)
    val_count = int(VAL_RATIO * n)
    test_count = n - train_count - val_count

    if n >= 3:
        if val_count == 0:
            val_count = 1
            train_count -= 1
        if test_count == 0:
            test_count = 1
            train_count -= 1
    if train_count < 1 and n > 0:
        train_count = 1
    while train_count + val_count + test_count > n:
        train_count -= 1
    test_count = n - train_count - val_count

    return (
        items[:train_count],
        items[train_count:train_count + val_count],
        items[train_count + val_count:],
    )


def grouped_stratified_split(group_items):
    strata = defaultdict(list)
    for group_key, filenames in group_items:
        strata[disease_for_group(filenames)].append((group_key, filenames))

    split_to_groups = {'train': [], 'valid': [], 'test': []}
    rng = random.Random(SEED)
    for disease, items in sorted(strata.items()):
        items = sorted(items, key=lambda item: item[0])
        rng.shuffle(items)
        train_items, val_items, test_items = split_one_stratum(items)
        split_to_groups['train'].extend(train_items)
        split_to_groups['valid'].extend(val_items)
        split_to_groups['test'].extend(test_items)
        print(
            f'  - {disease}: {len(train_items)} train groups, '
            f'{len(val_items)} valid groups, {len(test_items)} test groups'
        )

    for split in split_to_groups:
        split_to_groups[split] = sorted(split_to_groups[split], key=lambda item: item[0])
    return split_to_groups


def grouped_random_split(group_items):
    group_items = sorted(group_items, key=lambda item: item[0])
    random.Random(SEED).shuffle(group_items)
    n_groups = len(group_items)
    train_group_count = int(TRAIN_RATIO * n_groups)
    val_group_count = int(VAL_RATIO * n_groups)
    return {
        'train': group_items[:train_group_count],
        'valid': group_items[train_group_count:train_group_count + val_group_count],
        'test': group_items[train_group_count + val_group_count:],
    }


def split_summary(split_groups):
    group_diseases = Counter()
    image_diseases = Counter()
    for _, filenames in split_groups:
        group_diseases[disease_for_group(filenames)] += 1
        for filename in filenames:
            _, disease, _, _ = parse_shrimp_group_key(filename)
            image_diseases[disease] += 1
    return group_diseases, image_diseases


def split_grouped_by_shrimp():
    if REBUILD_SPLIT_FROM_ALL_SPLITS:
        image_paths = rebuild_train_pool_from_all_splits()
    else:
        image_paths = image_files_in_split('train')

    groups = defaultdict(list)
    disease_counts = Counter()
    unparsed = []

    for image_path in image_paths:
        group_key, disease, shrimp_id, img_num = parse_shrimp_group_key(image_path.name)
        groups[group_key].append(image_path.name)
        disease_counts[disease] += 1
        if disease == 'unparsed':
            unparsed.append(image_path.name)

    group_items = sorted(groups.items(), key=lambda item: item[0])
    if GROUP_STRATIFY_BY_DISEASE:
        print('Building shrimp-grouped, disease-stratified split:')
        split_to_groups = grouped_stratified_split(group_items)
    else:
        print('Building shrimp-grouped random split:')
        split_to_groups = grouped_random_split(group_items)

    for split, split_groups in split_to_groups.items():
        for _, filenames in split_groups:
            for filename in filenames:
                move_image_and_label(Path(base_path) / 'train' / 'images' / filename, split)

    print('Shrimp-grouped split complete:')
    for split, split_groups in split_to_groups.items():
        image_count = sum(len(filenames) for _, filenames in split_groups)
        group_diseases, image_diseases = split_summary(split_groups)
        print(f'  - {split}: {len(split_groups)} shrimp groups, {image_count} images')
        print(f'    group disease counts: {dict(sorted(group_diseases.items()))}')
        print(f'    image disease counts: {dict(sorted(image_diseases.items()))}')

    print('Source filename disease counts before split:', dict(sorted(disease_counts.items())))
    if unparsed:
        print(f'Warning: {len(unparsed)} filenames did not match the shrimp naming pattern. They were split as single-image groups.')
        print('First unparsed examples:', unparsed[:10])

    group_to_split = {}
    leakage = []
    for split in ['train', 'valid', 'test']:
        for image_path in image_files_in_split(split):
            group_key, *_ = parse_shrimp_group_key(image_path.name)
            previous_split = group_to_split.setdefault(group_key, split)
            if previous_split != split:
                leakage.append((group_key, previous_split, split, image_path.name))

    if leakage:
        raise RuntimeError(f'Shrimp-level split leakage detected: {leakage[:10]}')
    print('Shrimp-level leakage check passed.')
    remove_yolo_label_caches(base_path)


if GROUP_SPLIT_BY_SHRIMP:
    split_grouped_by_shrimp()
else:
    valid_images_dir = Path(base_path) / 'valid' / 'images'
    test_images_dir = Path(base_path) / 'test' / 'images'

    if not any(valid_images_dir.glob('*')) and not any(test_images_dir.glob('*')):
        image_files = sorted(
            f for f in os.listdir(os.path.join(train_path, 'images'))
            if f.lower().endswith(IMAGE_EXTENSIONS)
        )
        random.shuffle(image_files)

        train_count = int(0.8 * len(image_files))
        val_count = int(0.1 * len(image_files))
        val_files = image_files[train_count:train_count + val_count]
        test_files = image_files[train_count + val_count:]

        def move_files(files, target_split):
            for f in files:
                move_image_and_label(Path(train_path) / 'images' / f, target_split)

        move_files(val_files, 'valid')
        move_files(test_files, 'test')
        print(f"Image-level split complete: {len(image_files) - len(val_files) - len(test_files)} train, {len(val_files)} val, {len(test_files)} test")
        remove_yolo_label_caches(base_path)
    else:
        print('Existing valid/test split detected. Keeping downloaded split.')

In [ ]:
import importlib.util
import subprocess
import sys

import os
from pathlib import Path

base_path = str(DATASET_DIR)
data_yaml_path = os.path.join(base_path, 'data.yaml')

# Baseline model is defined in the run-control cell. Keep this smoke check
# synchronized with the paper-facing one-row baseline configuration.
YOLO_MODEL = YOLO_MODELS[0]
MODEL_STEM = Path(YOLO_MODEL).stem
RUN_BASE_NAME = f'{MODEL_STEM}_shrimp_seg_stratified_grouped_specimen'

# Load once here as a smoke check. Training cells instantiate fresh models per experiment.
model = YOLO(YOLO_MODEL)
print('Configured baseline segmentation model:')
for configured_model in YOLO_MODELS:
    print(f'  - {configured_model}')
print(f'Run name prefix: {RUN_BASE_NAME}')

In [ ]:
import yaml

# Update data.yaml to use correct paths
with open(data_yaml_path, 'r') as f:
    content = yaml.safe_load(f)

content['train'] = str(DATASET_DIR / 'train' / 'images')
content['val'] = str(DATASET_DIR / 'valid' / 'images')
content['test'] = str(DATASET_DIR / 'test' / 'images')

with open(data_yaml_path, 'w') as f:
    yaml.dump(content, f)

print("data.yaml updated with absolute paths.")

### Exploratory Data Analysis (EDA)
We will analyze the dataset to understand the class distribution and visualize some sample images with their masks.

In [ ]:
import os
import yaml
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter
from pathlib import Path

# Load class names from data.yaml. Empty label files are healthy shrimp negatives.
with open(data_yaml_path, 'r') as f:
    data_config = yaml.safe_load(f)

class_names = data_config.get('names', [])
HEALTHY_CLASS_NAME = 'healthy'
print(f"Disease mask classes found: {class_names}")
print(f"Empty label files will be treated as: {HEALTHY_CLASS_NAME} shrimp negatives")


def split_label_stats(label_dir):
    instance_counts = Counter()
    labeled_images = 0
    healthy_images = 0
    missing_or_empty = 0
    label_dir = Path(label_dir)
    for label_file in label_dir.glob('*.txt'):
        lines = [line.strip() for line in label_file.read_text().splitlines() if line.strip()]
        if not lines:
            healthy_images += 1
            continue
        labeled_images += 1
        for line in lines:
            class_id = int(float(line.split()[0]))
            instance_counts[class_id] += 1
    return {
        'instance_counts': instance_counts,
        'labeled_images': labeled_images,
        'healthy_images': healthy_images,
        'total_label_files': labeled_images + healthy_images,
    }


stats = {}
for split in ['train', 'valid', 'test']:
    label_dir = os.path.join(base_path, split, 'labels')
    stats[split] = split_label_stats(label_dir)

for split, split_stats in stats.items():
    print(f"\n{split.capitalize()} Split:")
    print(f"  - labeled disease images: {split_stats['labeled_images']}")
    print(f"  - healthy negative images: {split_stats['healthy_images']}")
    for cid, count in split_stats['instance_counts'].items():
        name = class_names[cid] if cid < len(class_names) else f"Unknown({cid})"
        print(f"  - {name}: {count} mask instances")

### Visualizing Class Imbalance
An imbalanced dataset can cause the model to be biased. Let's visualize the distribution across our splits.

In [ ]:
import pandas as pd
import seaborn as sns

instance_plot_data = []
image_plot_data = []
for split, split_stats in stats.items():
    image_plot_data.append({'Split': split, 'Class': HEALTHY_CLASS_NAME, 'Images': split_stats['healthy_images']})
    image_plot_data.append({'Split': split, 'Class': 'diseased_labeled', 'Images': split_stats['labeled_images']})
    for cid, count in split_stats['instance_counts'].items():
        instance_plot_data.append({'Split': split, 'Class': class_names[cid], 'Instances': count})

if instance_plot_data:
    df_instances = pd.DataFrame(instance_plot_data)
    plt.figure(figsize=(10, 6))
    sns.barplot(data=df_instances, x='Split', y='Instances', hue='Class')
    plt.title('Disease Mask Instance Distribution across Splits')
    plt.show()

if image_plot_data:
    df_images = pd.DataFrame(image_plot_data)
    plt.figure(figsize=(10, 6))
    sns.barplot(data=df_images, x='Split', y='Images', hue='Class')
    plt.title('Healthy Negative vs Diseased-Labeled Image Counts')
    plt.show()

for split, split_stats in stats.items():
    total_instances = sum(split_stats['instance_counts'].values())
    blackgill_ratio = (split_stats['instance_counts'].get(0, 0) / max(1, total_instances)) * 100
    healthy_ratio = (split_stats['healthy_images'] / max(1, split_stats['total_label_files'])) * 100
    print(f"{split.capitalize()}: {blackgill_ratio:.2f}% blackgill instances; {healthy_ratio:.2f}% healthy negative images")

## Training and Evaluation Helpers

In [ ]:
import json
import csv
import gc
import math
import shutil
import time
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import yaml
from IPython.display import display

EXPERIMENT_ROOT = Path(EXPERIMENT_ROOT)
RUNS_DIR = Path(RUNS_DIR)
REPORT_DIR = EXPERIMENT_ROOT / 'reports'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

SMOKE_RUN = bool(globals().get('SMOKE_RUN', False))
TRAIN_IMGSZ = 320 if SMOKE_RUN else 640
TRAIN_EPOCHS = 1 if SMOKE_RUN else 100
TRAIN_BATCH = 8 if SMOKE_RUN else 16
TRAIN_PATIENCE = 1 if SMOKE_RUN else 40
RUN_TEST_EVALUATION = not SMOKE_RUN

COUNT_PENALTY_WEIGHT = 0.05
DISEASE_MISS_PENALTY_WEIGHT = 0.15
HEALTHY_FP_PENALTY_WEIGHT = 0.10
PREDICT_CONF_FOR_COUNT = 0.25

ABLATION_MODES = [
    {
        'mode': 'W1',
        'enabled': True,
        'fourier_enabled': True,
        'clean_light_aug_enabled': False,
        'default_hook_enabled': False,
        'key': 'W1_fourier_highpass_s50_a0p10_no_aug_hook_off',
        'name': 'W1 | Fourier high-pass s50 a0.10 | all YOLO aug zero | hook off',
    },
]

EXPERIMENTS = [mode for mode in ABLATION_MODES if mode.get('enabled', True)]

NO_AUG_TRAIN_ARGS = {
    # No Fourier, no preprocessing, no explicit YOLO augmentation.
    # The hidden Albumentations hook is separately disabled before training.
    'auto_augment': None,
    'erasing': 0.0,
    'mosaic': 0.0,
    'mixup': 0.0,
    'cutmix': 0.0,
    'copy_paste': 0.0,
    'fliplr': 0.0,
    'flipud': 0.0,
    'hsv_h': 0.0,
    'hsv_s': 0.0,
    'hsv_v': 0.0,
    'degrees': 0.0,
    'translate': 0.0,
    'scale': 0.0,
    'shear': 0.0,
    'perspective': 0.0,
    'multi_scale': False,
    'bgr': 0.0,
    'close_mosaic': 0,
}


FOURIER_SIGMA = 50
FOURIER_ALPHA = 0.10
FOURIER_BATCH_SIZE = 4
USE_GPU_FOURIER = True
_FOURIER_LOWPASS_CACHE = {}


LIGHT_AUG_TRAIN_ARGS = {
    'auto_augment': None, 'erasing': 0.0, 'mosaic': 0.0, 'mixup': 0.0,
    'cutmix': 0.0, 'copy_paste': 0.0, 'fliplr': 0.5, 'flipud': 0.0,
    'hsv_h': 0.01, 'hsv_s': 0.35, 'hsv_v': 0.20, 'degrees': 0.0,
    'translate': 0.05, 'scale': 0.20, 'shear': 0.0, 'perspective': 0.0,
    'multi_scale': False, 'bgr': 0.0, 'close_mosaic': 0,
}


def train_args_for_mode(exp):
    return LIGHT_AUG_TRAIN_ARGS if exp.get('clean_light_aug_enabled') else NO_AUG_TRAIN_ARGS


def augmentation_policy_name(exp):
    return 'clean_light_yolo_aug' if exp.get('clean_light_aug_enabled') else 'none_all_yolo_aug_zero'


def fourier_device():
    if USE_GPU_FOURIER and torch.cuda.is_available():
        return torch.device('cuda')
    return torch.device('cpu')


def fourier_lowpass(height, width, sigma, device):
    key = (int(height), int(width), float(sigma), str(device))
    cached = _FOURIER_LOWPASS_CACHE.get(key)
    if cached is not None:
        return cached
    y = torch.arange(height, dtype=torch.float32, device=device) - height / 2.0
    x = torch.arange(width, dtype=torch.float32, device=device) - width / 2.0
    yy, xx = torch.meshgrid(y, x, indexing='ij')
    lowpass = torch.exp(-(xx * xx + yy * yy) / (2.0 * float(sigma) * float(sigma)))
    lowpass = lowpass[None, None, :, :]
    _FOURIER_LOWPASS_CACHE[key] = lowpass
    return lowpass


def fourier_highpass_boost_image_cpu(img, sigma=FOURIER_SIGMA, alpha=FOURIER_ALPHA):
    img_float = img.astype(np.float32)
    height, width = img_float.shape[:2]
    y = np.arange(height, dtype=np.float32) - height / 2.0
    x = np.arange(width, dtype=np.float32) - width / 2.0
    xx, yy = np.meshgrid(x, y)
    lowpass = np.exp(-(xx * xx + yy * yy) / (2.0 * float(sigma) * float(sigma))).astype(np.float32)
    enhanced_channels = []
    for channel_idx in range(img_float.shape[2]):
        channel = img_float[:, :, channel_idx]
        freq = np.fft.fftshift(np.fft.fft2(channel))
        low_freq = freq * lowpass
        low = np.fft.ifft2(np.fft.ifftshift(low_freq)).real
        high = channel - low
        enhanced_channels.append(channel + float(alpha) * high)
    enhanced = np.stack(enhanced_channels, axis=2)
    return np.clip(enhanced, 0, 255).astype(np.uint8)


def fourier_highpass_boost_batch_torch(imgs, sigma=FOURIER_SIGMA, alpha=FOURIER_ALPHA, device=None):
    if not imgs:
        return []
    if device is None:
        device = fourier_device()
    if device.type != 'cuda':
        return [fourier_highpass_boost_image_cpu(img, sigma=sigma, alpha=alpha) for img in imgs]

    arr = np.stack(imgs, axis=0).astype(np.float32)
    tensor = torch.from_numpy(arr).to(device=device, non_blocking=True).permute(0, 3, 1, 2).contiguous()
    height, width = tensor.shape[-2:]
    lowpass = fourier_lowpass(height, width, sigma, device)
    freq = torch.fft.fftshift(torch.fft.fft2(tensor, dim=(-2, -1)), dim=(-2, -1))
    low_freq = freq * lowpass
    low = torch.fft.ifft2(torch.fft.ifftshift(low_freq, dim=(-2, -1)), dim=(-2, -1)).real
    enhanced = torch.clamp(tensor + float(alpha) * (tensor - low), 0, 255)
    out = enhanced.permute(0, 2, 3, 1).byte().cpu().numpy()
    return [out[i] for i in range(out.shape[0])]


def apply_fourier_batch(image_batch, path_batch, sigma, alpha, device):
    if not image_batch:
        return {'gpu_images': 0, 'cpu_images': 0, 'oom_splits': 0}
    if device.type != 'cuda':
        enhanced_batch = [fourier_highpass_boost_image_cpu(img, sigma=sigma, alpha=alpha) for img in image_batch]
        for image_path, enhanced in zip(path_batch, enhanced_batch):
            cv2.imwrite(str(image_path), enhanced)
        return {'gpu_images': 0, 'cpu_images': len(image_batch), 'oom_splits': 0}

    try:
        enhanced_batch = fourier_highpass_boost_batch_torch(image_batch, sigma=sigma, alpha=alpha, device=device)
        for image_path, enhanced in zip(path_batch, enhanced_batch):
            cv2.imwrite(str(image_path), enhanced)
        return {'gpu_images': len(image_batch), 'cpu_images': 0, 'oom_splits': 0}
    except RuntimeError as exc:
        message = str(exc).lower()
        if 'out of memory' not in message and 'cuda' not in message:
            raise
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        if len(image_batch) == 1:
            print(f'GPU Fourier failed for one image ({exc}); falling back to CPU: {path_batch[0].name}')
            enhanced = fourier_highpass_boost_image_cpu(image_batch[0], sigma=sigma, alpha=alpha)
            cv2.imwrite(str(path_batch[0]), enhanced)
            return {'gpu_images': 0, 'cpu_images': 1, 'oom_splits': 1}

        mid = len(image_batch) // 2
        print(f'GPU Fourier OOM for batch of {len(image_batch)}; retrying as {mid} + {len(image_batch) - mid}.')
        left = apply_fourier_batch(image_batch[:mid], path_batch[:mid], sigma=sigma, alpha=alpha, device=device)
        right = apply_fourier_batch(image_batch[mid:], path_batch[mid:], sigma=sigma, alpha=alpha, device=device)
        return {
            'gpu_images': left['gpu_images'] + right['gpu_images'],
            'cpu_images': left['cpu_images'] + right['cpu_images'],
            'oom_splits': left['oom_splits'] + right['oom_splits'] + 1,
        }

def apply_fourier_highpass_to_image_dir(image_dir, sigma=FOURIER_SIGMA, alpha=FOURIER_ALPHA, batch_size=FOURIER_BATCH_SIZE):
    image_paths = []
    for ext in IMAGE_EXTENSIONS:
        image_paths.extend(Path(image_dir).glob(f'*{ext}'))
    image_paths = sorted(image_paths)
    device = fourier_device()
    if device.type == 'cuda':
        gc.collect()
        torch.cuda.empty_cache()
    print(
        f'Applying Fourier high-pass boost to {len(image_paths)} images in {image_dir} '
        f'(sigma={sigma}, alpha={alpha}, device={device}, batch_size={batch_size})'
    )

    failed = []
    image_batch = []
    path_batch = []
    batch_shape = None
    total_gpu_images = 0
    total_cpu_images = 0
    total_oom_splits = 0

    def flush_batch():
        nonlocal image_batch, path_batch, batch_shape, total_gpu_images, total_cpu_images, total_oom_splits
        if image_batch:
            stats = apply_fourier_batch(image_batch, path_batch, sigma=sigma, alpha=alpha, device=device)
            total_gpu_images += stats['gpu_images']
            total_cpu_images += stats['cpu_images']
            total_oom_splits += stats['oom_splits']
        image_batch = []
        path_batch = []
        batch_shape = None

    for image_path in image_paths:
        img = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
        if img is None:
            failed.append(str(image_path))
            continue
        if batch_shape is not None and img.shape != batch_shape:
            flush_batch()
        image_batch.append(img)
        path_batch.append(image_path)
        batch_shape = img.shape
        if len(image_batch) >= batch_size:
            flush_batch()
    flush_batch()

    if device.type == 'cuda':
        torch.cuda.empty_cache()
    if failed:
        print(f'Warning: failed to read {len(failed)} images. First examples: {failed[:5]}')
    print(f'Fourier split summary for {image_dir}: gpu_images={total_gpu_images}, cpu_images={total_cpu_images}, oom_splits={total_oom_splits}')
    return {
        'images': len(image_paths),
        'failed': len(failed),
        'device': str(device),
        'batch_size': int(batch_size),
        'gpu_images': int(total_gpu_images),
        'cpu_images': int(total_cpu_images),
        'oom_splits': int(total_oom_splits),
    }

def apply_fourier_highpass_to_dataset(dataset_dir, sigma=FOURIER_SIGMA, alpha=FOURIER_ALPHA):
    summary = {}
    for split in ['train', 'valid', 'test']:
        summary[split] = apply_fourier_highpass_to_image_dir(Path(dataset_dir) / split / 'images', sigma=sigma, alpha=alpha)
    remove_yolo_label_caches(dataset_dir)
    return summary

def configure_ultralytics_albumentations_hook(disable=False):
    """Disable or restore Ultralytics' default Albumentations hook for this run."""
    try:
        import importlib
        import ultralytics.data.augment as yolo_augment
    except Exception as exc:
        print(f'Could not access Ultralytics Albumentations hook: {exc}')
        return

    if not disable:
        # Important in notebooks: a previous cell/run may already have monkey-patched
        # Albumentations. Reloading restores the original Ultralytics class.
        importlib.reload(yolo_augment)
        print('Ultralytics Albumentations hook enabled/restored for this run.')
        return

    class NoOpAlbumentations:
        contains_spatial = False

        def __init__(self, *args, **kwargs):
            self.transform = None

        def __call__(self, labels):
            return labels

    yolo_augment.Albumentations = NoOpAlbumentations
    print('Ultralytics Albumentations hook disabled for this run.')


def remove_yolo_label_caches(root):
    for cache_path in Path(root).glob('**/*.cache'):
        cache_path.unlink()
        print(f'Removed stale cache: {cache_path}')



def find_image_for_label(image_dir, label_file):
    stem = Path(label_file).stem
    for ext in IMAGE_EXTENSIONS:
        candidate = Path(image_dir) / f'{stem}{ext}'
        if candidate.exists():
            return candidate
    return None

def count_labeled_images(label_dir):
    labeled = 0
    healthy = 0
    instances = 0
    for label_path in Path(label_dir).glob('*.txt'):
        lines = [line.strip() for line in label_path.read_text().splitlines() if line.strip()]
        if lines:
            labeled += 1
            instances += len(lines)
        else:
            healthy += 1
    return {'labeled_images': labeled, 'healthy_images': healthy, 'instances': instances}


def write_data_yaml(dataset_dir, yaml_path, val_dir='valid', test_dir='test'):
    with open(data_yaml_path, 'r') as f:
        content = yaml.safe_load(f)
    content['train'] = str(Path(dataset_dir) / 'train' / 'images')
    content['val'] = str(Path(dataset_dir) / val_dir / 'images')
    content['test'] = str(Path(dataset_dir) / test_dir / 'images')
    with open(yaml_path, 'w') as f:
        yaml.safe_dump(content, f, sort_keys=False)
    return yaml_path


def copy_dataset_for_experiment(exp_key):
    src = Path(base_path)
    dst = EXPERIMENT_ROOT / exp_key / 'dataset'
    if dst.exists():
        shutil.rmtree(dst)
    ignore = shutil.ignore_patterns('runs', '*.cache')
    shutil.copytree(src, dst, ignore=ignore)
    remove_yolo_label_caches(dst)
    return dst


def copy_split_by_label_state(src_dataset, dst_dataset, split, want_labeled):
    src_images = Path(src_dataset) / split / 'images'
    src_labels = Path(src_dataset) / split / 'labels'
    dst_images = Path(dst_dataset) / split / 'images'
    dst_labels = Path(dst_dataset) / split / 'labels'
    dst_images.mkdir(parents=True, exist_ok=True)
    dst_labels.mkdir(parents=True, exist_ok=True)

    copied = 0
    for label_path in sorted(src_labels.glob('*.txt')):
        lines = [line.strip() for line in label_path.read_text().splitlines() if line.strip()]
        is_labeled = bool(lines)
        if is_labeled != want_labeled:
            continue
        image_path = find_image_for_label(src_images, label_path.name)
        if image_path is None:
            continue
        shutil.copy2(image_path, dst_images / image_path.name)
        shutil.copy2(label_path, dst_labels / label_path.name)
        copied += 1
    return copied


def make_state_eval_dataset(src_dataset, exp_key, state_name, want_labeled):
    dst = EXPERIMENT_ROOT / exp_key / f'dataset_{state_name}_eval'
    if dst.exists():
        shutil.rmtree(dst)

    for sub in ['images', 'labels']:
        (dst / 'train' / sub).mkdir(parents=True, exist_ok=True)
    copied = {}
    for split in ['valid', 'test']:
        copied[split] = copy_split_by_label_state(src_dataset, dst, split, want_labeled=want_labeled)
    yaml_path = dst / f'data_{state_name}.yaml'
    write_data_yaml(dst, yaml_path)
    print(f'{state_name} eval dataset for {exp_key}: {copied}')
    return dst, yaml_path, copied


def make_labeled_only_eval_dataset(src_dataset, exp_key):
    return make_state_eval_dataset(src_dataset, exp_key, 'labeled_only', want_labeled=True)


def make_healthy_only_eval_dataset(src_dataset, exp_key):
    return make_state_eval_dataset(src_dataset, exp_key, 'healthy_only', want_labeled=False)


def metric_value(metrics, dotted_path, default=float('nan')):
    obj = metrics
    for part in dotted_path.split('.'):
        if not hasattr(obj, part):
            return default
        obj = getattr(obj, part)
    try:
        return float(obj)
    except Exception:
        return default


def count_prediction_errors(model, images_dir, labels_dir, conf=PREDICT_CONF_FOR_COUNT):
    image_paths = []
    for ext in IMAGE_EXTENSIONS:
        image_paths.extend(Path(images_dir).glob(f'*{ext}'))
    image_paths = sorted(image_paths)
    if not image_paths:
        return {
            'images': 0,
            'gt_total': 0,
            'pred_box_total': 0,
            'pred_mask_total': 0,
            'box_count_mae': float('nan'),
            'mask_count_mae': float('nan'),
            'box_count_exact': float('nan'),
            'mask_count_exact': float('nan'),
            'disease_images': 0,
            'disease_box_miss_images': 0,
            'disease_mask_miss_images': 0,
            'disease_box_miss_rate': float('nan'),
            'disease_mask_miss_rate': float('nan'),
        }

    results = model.predict(source=[str(p) for p in image_paths], imgsz=TRAIN_IMGSZ, conf=conf, verbose=False)
    box_errors = []
    mask_errors = []
    box_exact = []
    mask_exact = []
    gt_total = 0
    pred_box_total = 0
    pred_mask_total = 0
    disease_images = 0
    disease_box_miss_images = 0
    disease_mask_miss_images = 0

    for image_path, result in zip(image_paths, results):
        label_path = Path(labels_dir) / f'{image_path.stem}.txt'
        gt_count = 0
        if label_path.exists():
            gt_count = len([line for line in label_path.read_text().splitlines() if line.strip()])
        box_count = len(result.boxes) if result.boxes is not None else 0
        mask_count = len(result.masks) if result.masks is not None else 0
        denom = max(1, gt_count)
        box_errors.append(abs(box_count - gt_count) / denom)
        mask_errors.append(abs(mask_count - gt_count) / denom)
        box_exact.append(float(box_count == gt_count))
        mask_exact.append(float(mask_count == gt_count))
        if gt_count > 0:
            disease_images += 1
            disease_box_miss_images += int(box_count == 0)
            disease_mask_miss_images += int(mask_count == 0)
        gt_total += gt_count
        pred_box_total += box_count
        pred_mask_total += mask_count

    disease_box_miss_rate = disease_box_miss_images / disease_images if disease_images else float('nan')
    disease_mask_miss_rate = disease_mask_miss_images / disease_images if disease_images else float('nan')
    return {
        'images': len(image_paths),
        'gt_total': gt_total,
        'pred_box_total': pred_box_total,
        'pred_mask_total': pred_mask_total,
        'box_count_mae': sum(box_errors) / len(box_errors),
        'mask_count_mae': sum(mask_errors) / len(mask_errors),
        'box_count_exact': sum(box_exact) / len(box_exact),
        'mask_count_exact': sum(mask_exact) / len(mask_exact),
        'disease_images': disease_images,
        'disease_box_miss_images': disease_box_miss_images,
        'disease_mask_miss_images': disease_mask_miss_images,
        'disease_box_miss_rate': disease_box_miss_rate,
        'disease_mask_miss_rate': disease_mask_miss_rate,
    }


def healthy_false_positive_summary(model, images_dir, conf=PREDICT_CONF_FOR_COUNT):
    image_paths = []
    for ext in IMAGE_EXTENSIONS:
        image_paths.extend(Path(images_dir).glob(f'*{ext}'))
    image_paths = sorted(image_paths)
    if not image_paths:
        return {
            'healthy_images': 0,
            'healthy_images_with_box_fp': 0,
            'healthy_images_with_mask_fp': 0,
            'healthy_box_fp_rate': float('nan'),
            'healthy_mask_fp_rate': float('nan'),
            'healthy_fp_boxes_total': 0,
            'healthy_fp_masks_total': 0,
            'healthy_fp_boxes_per_image': float('nan'),
            'healthy_fp_masks_per_image': float('nan'),
            'healthy_avg_fp_confidence': float('nan'),
        }

    results = model.predict(source=[str(p) for p in image_paths], imgsz=TRAIN_IMGSZ, conf=conf, verbose=False)
    images_with_box_fp = 0
    images_with_mask_fp = 0
    box_total = 0
    mask_total = 0
    confidences = []

    for result in results:
        box_count = len(result.boxes) if result.boxes is not None else 0
        mask_count = len(result.masks) if result.masks is not None else 0
        if box_count > 0:
            images_with_box_fp += 1
            try:
                confidences.extend([float(v) for v in result.boxes.conf.detach().cpu().tolist()])
            except Exception:
                pass
        if mask_count > 0:
            images_with_mask_fp += 1
        box_total += box_count
        mask_total += mask_count

    n = len(image_paths)
    return {
        'healthy_images': n,
        'healthy_images_with_box_fp': images_with_box_fp,
        'healthy_images_with_mask_fp': images_with_mask_fp,
        'healthy_box_fp_rate': images_with_box_fp / n,
        'healthy_mask_fp_rate': images_with_mask_fp / n,
        'healthy_fp_boxes_total': box_total,
        'healthy_fp_masks_total': mask_total,
        'healthy_fp_boxes_per_image': box_total / n,
        'healthy_fp_masks_per_image': mask_total / n,
        'healthy_avg_fp_confidence': sum(confidences) / len(confidences) if confidences else 0.0,
    }


def healthy_aware_score(labeled_map50, count_summary, healthy_fp_summary):
    count_penalty = COUNT_PENALTY_WEIGHT * count_summary['mask_count_mae']
    disease_miss_penalty = DISEASE_MISS_PENALTY_WEIGHT * count_summary['disease_box_miss_rate']
    healthy_fp_penalty = HEALTHY_FP_PENALTY_WEIGHT * healthy_fp_summary['healthy_mask_fp_rate']
    return labeled_map50 - count_penalty - disease_miss_penalty - healthy_fp_penalty


def read_best_epoch_from_results(run_path):
    results_csv = Path(run_path) / 'results.csv'
    if not results_csv.exists():
        return {}
    df = pd.read_csv(results_csv)
    df.columns = [c.strip() for c in df.columns]
    mask_col = 'metrics/mAP50(M)'
    if mask_col not in df.columns:
        return {'epochs_ran': len(df)}
    best_idx = df[mask_col].idxmax()
    first = df.iloc[0]
    best = df.iloc[best_idx]
    last = df.iloc[-1]
    return {
        'epochs_ran': int(len(df)),
        'best_epoch_by_mask_map50': int(best['epoch']) if 'epoch' in df.columns else int(best_idx + 1),
        'first_train_seg_loss': float(first.get('train/seg_loss', float('nan'))),
        'best_val_mask_map50': float(best.get(mask_col, float('nan'))),
        'best_val_mask_map50_95': float(best.get('metrics/mAP50-95(M)', float('nan'))),
        'last_val_mask_map50': float(last.get(mask_col, float('nan'))),
        'last_val_mask_map50_95': float(last.get('metrics/mAP50-95(M)', float('nan'))),
        'last_train_seg_loss': float(last.get('train/seg_loss', float('nan'))),
        'last_val_seg_loss': float(last.get('val/seg_loss', float('nan'))),
        'seg_loss_gap_val_minus_train': float(last.get('val/seg_loss', float('nan')) - last.get('train/seg_loss', float('nan'))),
    }


def run_experiment(exp):
    print('\n' + '#' * 90)
    print(f"Starting experiment: {exp['name']}")
    print('#' * 90)

    dataset_dir = copy_dataset_for_experiment(exp['key'])
    if exp.get('fourier_enabled'):
        fourier_transform_summary = apply_fourier_highpass_to_dataset(dataset_dir, sigma=FOURIER_SIGMA, alpha=FOURIER_ALPHA)
    else:
        fourier_transform_summary = {'strategy': 'fourier_disabled', 'fourier_enabled': False, 'sigma': None, 'alpha': None}
        print('Fourier disabled for this mode; dataset images remain original.')

    remove_yolo_label_caches(dataset_dir)
    yaml_path = dataset_dir / 'data.yaml'
    write_data_yaml(dataset_dir, yaml_path)
    labeled_eval_dir, labeled_eval_yaml, _ = make_labeled_only_eval_dataset(dataset_dir, exp['key'])
    healthy_eval_dir, healthy_eval_yaml, _ = make_healthy_only_eval_dataset(dataset_dir, exp['key'])

    split_counts = {}
    actual_image_files = {}
    for split in ['train', 'valid', 'test']:
        split_counts[split] = count_labeled_images(dataset_dir / split / 'labels')
        image_dir = dataset_dir / split / 'images'
        actual_image_files[split] = sum(1 for ext in IMAGE_EXTENSIONS for image_path in image_dir.glob(f'*{ext}') if image_path.is_file())
        print(f'{exp["key"]} {split}: {split_counts[split]} | actual_image_files={actual_image_files[split]}')

    training_seed = int(globals().get('CURRENT_RUN_SEED', 42))
    run_name = f'{RUN_BASE_NAME}_seed{training_seed}_{exp["key"]}' + ('_smoke' if SMOKE_RUN else '')
    hook_disabled = not bool(exp.get('default_hook_enabled'))
    configure_ultralytics_albumentations_hook(disable=hook_disabled)
    yolo = YOLO(YOLO_MODEL)
    start = time.time()
    yolo.train(data=str(yaml_path), task='segment', imgsz=TRAIN_IMGSZ, epochs=TRAIN_EPOCHS, batch=TRAIN_BATCH, patience=TRAIN_PATIENCE, seed=training_seed, project=str(RUNS_DIR), name=run_name, exist_ok=True, pretrained=True, plots=not SMOKE_RUN, verbose=True, **train_args_for_mode(exp))
    train_time_min = (time.time() - start) / 60

    run_path = RUNS_DIR / run_name
    best_path = run_path / 'weights' / 'best.pt'
    best_model = YOLO(str(best_path))

    full_val = best_model.val(data=str(yaml_path), split='val', imgsz=TRAIN_IMGSZ, plots=not SMOKE_RUN, verbose=False)
    labeled_val = best_model.val(data=str(labeled_eval_yaml), split='val', imgsz=TRAIN_IMGSZ, plots=False, verbose=False)
    labeled_val_count = count_prediction_errors(best_model, labeled_eval_dir / 'valid' / 'images', labeled_eval_dir / 'valid' / 'labels')
    healthy_val_fp = healthy_false_positive_summary(best_model, healthy_eval_dir / 'valid' / 'images')
    labeled_val_map50 = metric_value(labeled_val, 'seg.map50')
    val_score = healthy_aware_score(labeled_val_map50, labeled_val_count, healthy_val_fp)

    row = {
        'experiment': exp['key'], 'mode': exp.get('mode'), 'name': exp['name'], 'model': YOLO_MODEL,
        'run_name': run_name, 'run_path': str(run_path), 'best_pt': str(best_path), 'smoke_run': SMOKE_RUN,
        'training_seed': training_seed, 'fourier_enabled': bool(exp.get('fourier_enabled')),
        'clean_light_aug_enabled': bool(exp.get('clean_light_aug_enabled')), 'default_hook_enabled': bool(exp.get('default_hook_enabled')),
        'baseline_augmentation_policy': augmentation_policy_name(exp), 'hidden_albumentations_disabled': hook_disabled,
        'preprocessing_policy': 'fourier_highpass' if exp.get('fourier_enabled') else 'none',
        'fourier_policy': 'highpass_s50_a0p10_all_splits' if exp.get('fourier_enabled') else 'none',
        'fourier_sigma': FOURIER_SIGMA if exp.get('fourier_enabled') else None, 'fourier_alpha': FOURIER_ALPHA if exp.get('fourier_enabled') else None,
        'ultralytics_version': ultralytics.__version__, 'train_epochs_requested': TRAIN_EPOCHS, 'train_patience': TRAIN_PATIENCE,
        'actual_train_image_files': actual_image_files.get('train'), 'actual_valid_image_files': actual_image_files.get('valid'), 'actual_test_image_files': actual_image_files.get('test'),
        'fourier_transform_summary': json.dumps(fourier_transform_summary), 'train_args': json.dumps(train_args_for_mode(exp)), 'train_time_min': round(train_time_min, 2),
        'full_val_box_map50': metric_value(full_val, 'box.map50'), 'full_val_mask_map50': metric_value(full_val, 'seg.map50'),
        'labeled_val_box_map50': metric_value(labeled_val, 'box.map50'), 'labeled_val_mask_map50': labeled_val_map50,
        'labeled_val_mask_map50_95': metric_value(labeled_val, 'seg.map'), 'labeled_val_gt_instances': labeled_val_count['gt_total'],
        'labeled_val_pred_boxes': labeled_val_count['pred_box_total'], 'labeled_val_pred_masks': labeled_val_count['pred_mask_total'],
        'labeled_val_mask_count_mae': labeled_val_count['mask_count_mae'], 'labeled_val_disease_box_miss_rate': labeled_val_count['disease_box_miss_rate'],
        'labeled_val_disease_mask_miss_rate': labeled_val_count['disease_mask_miss_rate'], 'healthy_val_images': healthy_val_fp['healthy_images'],
        'healthy_val_mask_fp_rate': healthy_val_fp['healthy_mask_fp_rate'], 'healthy_val_fp_masks_per_image': healthy_val_fp['healthy_fp_masks_per_image'],
        'healthy_aware_labeled_val_mask_map50': val_score,
    }

    if RUN_TEST_EVALUATION:
        full_test = best_model.val(data=str(yaml_path), split='test', imgsz=TRAIN_IMGSZ, plots=True, verbose=False)
        labeled_test = best_model.val(data=str(labeled_eval_yaml), split='test', imgsz=TRAIN_IMGSZ, plots=False, verbose=False)
        test_count = count_prediction_errors(best_model, dataset_dir / 'test' / 'images', dataset_dir / 'test' / 'labels')
        labeled_test_count = count_prediction_errors(best_model, labeled_eval_dir / 'test' / 'images', labeled_eval_dir / 'test' / 'labels')
        healthy_test_fp = healthy_false_positive_summary(best_model, healthy_eval_dir / 'test' / 'images')
        labeled_test_map50 = metric_value(labeled_test, 'seg.map50')
        row.update({
            'full_test_box_map50': metric_value(full_test, 'box.map50'), 'full_test_box_map50_95': metric_value(full_test, 'box.map'),
            'full_test_mask_map50': metric_value(full_test, 'seg.map50'), 'full_test_mask_map50_95': metric_value(full_test, 'seg.map'),
            'labeled_test_box_map50': metric_value(labeled_test, 'box.map50'), 'labeled_test_mask_map50': labeled_test_map50,
            'labeled_test_mask_map50_95': metric_value(labeled_test, 'seg.map'), 'test_gt_instances': test_count['gt_total'],
            'test_pred_boxes': test_count['pred_box_total'], 'test_pred_masks': test_count['pred_mask_total'], 'test_mask_count_mae': test_count['mask_count_mae'],
            'labeled_test_gt_instances': labeled_test_count['gt_total'], 'labeled_test_pred_boxes': labeled_test_count['pred_box_total'],
            'labeled_test_pred_masks': labeled_test_count['pred_mask_total'], 'labeled_test_mask_count_mae': labeled_test_count['mask_count_mae'],
            'labeled_test_mask_count_exact': labeled_test_count['mask_count_exact'], 'labeled_test_disease_box_miss_rate': labeled_test_count['disease_box_miss_rate'],
            'labeled_test_disease_mask_miss_rate': labeled_test_count['disease_mask_miss_rate'], 'healthy_test_images': healthy_test_fp['healthy_images'],
            'healthy_test_mask_fp_rate': healthy_test_fp['healthy_mask_fp_rate'], 'healthy_test_box_fp_rate': healthy_test_fp['healthy_box_fp_rate'],
            'healthy_test_fp_masks_total': healthy_test_fp['healthy_fp_masks_total'], 'healthy_test_fp_masks_per_image': healthy_test_fp['healthy_fp_masks_per_image'],
            'healthy_test_avg_fp_confidence': healthy_test_fp['healthy_avg_fp_confidence'],
            'healthy_aware_labeled_test_mask_map50': healthy_aware_score(labeled_test_map50, labeled_test_count, healthy_test_fp),
        })

    row.update(read_best_epoch_from_results(run_path))
    del yolo, best_model
    gc.collect()
    return row


train_args_json = REPORT_DIR / 'train_args_048_w1_train_aug_test_tta_seed42.json'
train_args_payload = {'fourier_sigma': FOURIER_SIGMA, 'fourier_alpha': FOURIER_ALPHA, 'no_aug_train_args': NO_AUG_TRAIN_ARGS, 'light_aug_train_args': LIGHT_AUG_TRAIN_ARGS, 'ablation_modes': ABLATION_MODES, 'enabled_modes': EXPERIMENTS, 'train_epochs_requested': TRAIN_EPOCHS, 'train_patience': TRAIN_PATIENCE}
train_args_json.write_text(json.dumps(train_args_payload, indent=2), encoding='utf-8')
print('Saved strict W1 train args:', train_args_json)


## Train Strict W1 and Evaluate the Canonical Split


In [ ]:
from collections import defaultdict
import gc
import hashlib
import random
from pathlib import Path

import pandas as pd
from IPython.display import display

# Override the baseline helper constants for this paper matrix.
EXPERIMENT_ROOT = Path(EXPERIMENT_ROOT)
RUNS_DIR = Path(RUNS_DIR)
REPORT_DIR = EXPERIMENT_ROOT / 'reports'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_IMGSZ = 320 if SMOKE_RUN else 640
TRAIN_EPOCHS = 1 if SMOKE_RUN else 100
TRAIN_BATCH = 8 if SMOKE_RUN else 16
TRAIN_PATIENCE = 1 if SMOKE_RUN else 40
RUN_TEST_EVALUATION = not SMOKE_RUN
PREDICT_CONF_FOR_COUNT = 0.25


def disease_stratum_for_image(image_path):
    _, disease, _, _ = parse_shrimp_group_key(Path(image_path).name)
    return str(disease).lower()


def image_split_from_strata(items_by_stratum, seed):
    rng = random.Random(seed)
    split_to_items = {'train': [], 'valid': [], 'test': []}

    for _, items in sorted(items_by_stratum.items()):
        items = list(items)
        rng.shuffle(items)
        n = len(items)
        n_train = int(round(n * TRAIN_RATIO))
        n_valid = int(round(n * VAL_RATIO))
        n_train = min(n_train, n)
        n_valid = min(n_valid, max(0, n - n_train))

        split_to_items['train'].extend(items[:n_train])
        split_to_items['valid'].extend(items[n_train:n_train + n_valid])
        split_to_items['test'].extend(items[n_train + n_valid:])

    return split_to_items


def plain_random_image_split(seed=42):
    # Deterministic naive image-level split. This intentionally allows specimen leakage
    # and does not disease-stratify, but it no longer depends on filesystem list order.
    rebuild_train_pool_from_all_splits()

    train_images_dir = Path(base_path) / 'train' / 'images'
    rng = random.Random(seed)
    image_files = sorted(
        f for f in os.listdir(train_images_dir)
        if f.lower().endswith(IMAGE_EXTENSIONS)
    )
    if not image_files:
        raise RuntimeError('No images found in train pool after rebuilding all splits.')

    rng.shuffle(image_files)

    train_count = int(TRAIN_RATIO * len(image_files))
    valid_count = int(VAL_RATIO * len(image_files))

    split_to_files = {
        'train': image_files[:train_count],
        'valid': image_files[train_count:train_count + valid_count],
        'test': image_files[train_count + valid_count:],
    }

    for split_name in ['valid', 'test']:
        for filename in split_to_files[split_name]:
            move_image_and_label(train_images_dir / filename, split_name)

    remove_yolo_label_caches(base_path)
    print(
        'Applied deterministic naive random image split: '
        f"{len(split_to_files['train'])} train, "
        f"{len(split_to_files['valid'])} valid, "
        f"{len(split_to_files['test'])} test"
    )
    return split_records()


def stratified_random_image_split(seed=42):
    # Intentionally image-level. This is the leakage-inflated reference protocol.
    rebuild_train_pool_from_all_splits()
    images = image_files_in_split('train')
    if not images:
        raise RuntimeError('No images found in train pool after rebuilding all splits.')

    by_stratum = defaultdict(list)
    for image_path in images:
        by_stratum[disease_stratum_for_image(image_path)].append(image_path)

    split_to_images = image_split_from_strata(by_stratum, seed)
    for split_name, split_images in split_to_images.items():
        for image_path in split_images:
            move_image_and_label(image_path, split_name)

    remove_yolo_label_caches(base_path)
    print('Applied stratified random image split.')
    return split_records()


def stratified_grouped_specimen_split(seed=42):
    # Fair protocol inherited from the clean baseline notebook.
    global SEED, GROUP_SPLIT_BY_SHRIMP, GROUP_STRATIFY_BY_DISEASE, REBUILD_SPLIT_FROM_ALL_SPLITS
    SEED = seed
    GROUP_SPLIT_BY_SHRIMP = True
    GROUP_STRATIFY_BY_DISEASE = True
    REBUILD_SPLIT_FROM_ALL_SPLITS = True
    split_grouped_by_shrimp()
    remove_yolo_label_caches(base_path)
    print('Applied stratified grouped-specimen split.')
    return split_records()


def apply_split_policy(policy_key, seed=42):
    if policy_key == 'plain_random_image':
        return plain_random_image_split(seed=seed)
    if policy_key == 'stratified_random_image':
        return stratified_random_image_split(seed=seed)
    if policy_key == 'stratified_grouped_specimen':
        return stratified_grouped_specimen_split(seed=seed)
    raise ValueError(f'Unknown split policy: {policy_key}')


def split_records():
    records = []
    for split_name in ['train', 'valid', 'test']:
        for image_path in image_files_in_split(split_name):
            group_key, disease, shrimp_id, img_num = parse_shrimp_group_key(image_path.name)
            label_path = Path(base_path) / split_name / 'labels' / f'{image_path.stem}.txt'
            label_lines = []
            class_ids = []
            if label_path.exists():
                label_lines = [line.strip() for line in label_path.read_text().splitlines() if line.strip()]
                for line in label_lines:
                    parts = line.split()
                    if parts:
                        class_ids.append(int(float(parts[0])))
            unique_class_ids = sorted(set(class_ids))
            records.append({
                'split': split_name,
                'image': image_path.name,
                'group_key': group_key,
                'disease': str(disease).lower(),
                'shrimp_id': shrimp_id,
                'mask_instances': len(label_lines),
                'is_labeled': bool(label_lines),
                'class_ids': ','.join(str(cid) for cid in unique_class_ids),
                'has_bg_mask': 0 in unique_class_ids,
                'has_wssv_mask': 1 in unique_class_ids,
                'is_coinfection': len(unique_class_ids) >= 2,
            })
    return pd.DataFrame(records)


def save_split_artifacts(policy_key, seed):
    df = split_records().sort_values(['split', 'image']).reset_index(drop=True)
    manifest_dir = REPORT_DIR / 'split_manifests'
    manifest_dir.mkdir(parents=True, exist_ok=True)
    prefix = f'{policy_key}_seed{seed}'

    manifest_csv = manifest_dir / f'{prefix}_manifest.csv'
    summary_csv = manifest_dir / f'{prefix}_summary.csv'
    fingerprint_txt = manifest_dir / f'{prefix}_fingerprint.txt'

    df.to_csv(manifest_csv, index=False)
    summary_rows = []
    split_hashes = []
    for split in ['train', 'valid', 'test']:
        split_df = df[df['split'] == split]
        names = split_df['image'].tolist()
        split_hash = hashlib.sha256('\n'.join(names).encode()).hexdigest()
        split_hashes.append(split_hash)
        summary_rows.append({
            'split': split,
            'images': int(len(split_df)),
            'specimens': int(split_df['group_key'].nunique()),
            'labeled_images': int(split_df['is_labeled'].sum()),
            'healthy_empty_images': int((~split_df['is_labeled']).sum()),
            'bg_mask_images': int(split_df['has_bg_mask'].sum()),
            'wssv_mask_images': int(split_df['has_wssv_mask'].sum()),
            'coinfection_images': int(split_df['is_coinfection'].sum()),
            'mask_instances': int(split_df['mask_instances'].sum()),
            'split_sha256': split_hash,
        })
    pd.DataFrame(summary_rows).to_csv(summary_csv, index=False)
    fingerprint = hashlib.sha256('||'.join(split_hashes).encode()).hexdigest()
    fingerprint_txt.write_text(fingerprint + '\n', encoding='utf-8')
    print(f'Saved split manifest: {manifest_csv}')
    print(f'Saved split summary: {summary_csv}')
    print(f'Split fingerprint: {fingerprint}')
    return {
        'split_manifest_csv': str(manifest_csv),
        'split_summary_csv': str(summary_csv),
        'split_fingerprint': fingerprint,
    }


def split_metadata(policy_key, model_name, seed):
    df = split_records()
    group_sets = {
        split: set(df.loc[df['split'] == split, 'group_key'])
        for split in ['train', 'valid', 'test']
    }

    row = {
        'split_policy': policy_key,
        'model': model_name,
        'seed': seed,
        'train_images': int((df['split'] == 'train').sum()),
        'valid_images': int((df['split'] == 'valid').sum()),
        'test_images': int((df['split'] == 'test').sum()),
        'train_specimens': len(group_sets['train']),
        'valid_specimens': len(group_sets['valid']),
        'test_specimens': len(group_sets['test']),
        'train_test_group_overlap': len(group_sets['train'] & group_sets['test']),
        'train_valid_group_overlap': len(group_sets['train'] & group_sets['valid']),
        'valid_test_group_overlap': len(group_sets['valid'] & group_sets['test']),
    }

    for split in ['train', 'valid', 'test']:
        split_df = df[df['split'] == split]
        row[f'{split}_labeled_images'] = int(split_df['is_labeled'].sum())
        row[f'{split}_healthy_empty_images'] = int((~split_df['is_labeled']).sum())
        row[f'{split}_bg_mask_images'] = int(split_df['has_bg_mask'].sum())
        row[f'{split}_wssv_mask_images'] = int(split_df['has_wssv_mask'].sum())
        row[f'{split}_coinfection_images'] = int(split_df['is_coinfection'].sum())
        row[f'{split}_mask_instances'] = int(split_df['mask_instances'].sum())

    return row


def count_model_params_million(checkpoint_name):
    yolo_for_count = YOLO(checkpoint_name)
    try:
        params = sum(p.numel() for p in yolo_for_count.model.parameters())
        return round(params / 1_000_000, 3)
    finally:
        del yolo_for_count
        gc.collect()


EXPERIMENTS = [mode for mode in ABLATION_MODES if mode.get('enabled', True)]
print('Enabled strict W1 training modes:', [mode['mode'] for mode in EXPERIMENTS])

paper_rows = []

for seed in SEEDS:
    for split_policy in SPLIT_POLICIES:
        print()
        print('=' * 100)
        print(f"Preparing split policy: {split_policy['key']} | seed={seed}")
        print(split_policy['description'])
        print('=' * 100)

        CURRENT_RUN_SEED = seed
        apply_split_policy(split_policy['key'], seed=seed)
        split_artifacts = save_split_artifacts(split_policy['key'], seed)

        for model_name in YOLO_MODELS:
            YOLO_MODEL = model_name
            MODEL_STEM = Path(YOLO_MODEL).stem
            RUN_BASE_NAME = f'{MODEL_STEM}_shrimp_seg_{split_policy["key"]}'
            params_million = count_model_params_million(YOLO_MODEL)

            for experiment in EXPERIMENTS:
                exp = dict(experiment)
                exp['key'] = f'{split_policy["key"]}_{MODEL_STEM}_{experiment["key"]}'
                exp['name'] = f'{split_policy["name"]} | {MODEL_STEM} | {experiment["name"]}'

                meta = split_metadata(split_policy['key'], YOLO_MODEL, seed)
                meta.update(split_artifacts)
                meta['params_million'] = params_million
                meta['split_policy_description'] = split_policy['description']

                result = run_experiment(exp)
                result.update(meta)
                paper_rows.append(result)

                partial_df = pd.DataFrame(paper_rows)
                partial_csv = REPORT_DIR / '048_w1_train_partial.csv'
                partial_df.to_csv(partial_csv, index=False)
                display(partial_df)
                print(f'Saved partial CSV: {partial_csv}')

summary_df = pd.DataFrame(paper_rows)
summary_df = summary_df.sort_values(
    ['split_policy', 'healthy_aware_labeled_val_mask_map50', 'labeled_val_mask_map50'],
    ascending=[True, False, False],
).reset_index(drop=True)

summary_csv = REPORT_DIR / '048_w1_train_summary.csv'
summary_df.to_csv(summary_csv, index=False)
print(f'Saved W1 training summary CSV: {summary_csv}')
display(summary_df)


## Compact W1 Training Row


In [ ]:
paper_cols = [
    'split_policy',
    'mode',
    'model',
    'seed',
    'training_seed',
    'params_million',
    'ultralytics_version',
    'fourier_enabled',
    'clean_light_aug_enabled',
    'default_hook_enabled',
    'preprocessing_policy',
    'fourier_policy',
    'fourier_sigma',
    'fourier_alpha',
    'baseline_augmentation_policy',
    'hidden_albumentations_disabled',
    'train_epochs_requested',
    'train_patience',
    'train_images',
    'valid_images',
    'test_images',
    'actual_train_image_files',
    'actual_valid_image_files',
    'actual_test_image_files',
    'train_coinfection_images',
    'valid_coinfection_images',
    'test_coinfection_images',
    'train_bg_mask_images',
    'valid_bg_mask_images',
    'test_bg_mask_images',
    'train_wssv_mask_images',
    'valid_wssv_mask_images',
    'test_wssv_mask_images',
    'train_specimens',
    'valid_specimens',
    'test_specimens',
    'train_test_group_overlap',
    'valid_test_group_overlap',
    'split_manifest_csv',
    'split_fingerprint',
    'epochs_ran',
    'best_epoch_by_mask_map50',
    'train_time_min',
    'full_test_mask_map50',
    'full_test_mask_map50_95',
    'labeled_test_mask_map50',
    'labeled_test_mask_map50_95',
    'healthy_test_mask_fp_rate',
    'healthy_test_fp_masks_per_image',
    'labeled_test_disease_mask_miss_rate',
    'labeled_test_mask_count_mae',
    'healthy_aware_labeled_val_mask_map50',
    'healthy_aware_labeled_test_mask_map50',
    'best_pt',
    'run_path',
]

available_paper_cols = [c for c in paper_cols if c in summary_df.columns]
paper_table = summary_df[available_paper_cols].copy()
paper_table_csv = REPORT_DIR / '048_w1_train_paper_row.csv'
paper_table.to_csv(paper_table_csv, index=False)

print(f'Saved compact W1 training paper row: {paper_table_csv}')
display(paper_table)

fair_rows = paper_table[paper_table['split_policy'] == 'stratified_grouped_specimen']
if not fair_rows.empty:
    print('Strict W1 grouped-specimen checkpoint:')
    display(fair_rows.sort_values('healthy_aware_labeled_val_mask_map50', ascending=False).head(3))


## Resolve the Newly Trained W1 Model


In [ ]:
# Resolve the newly trained W1 checkpoint and its Fourier-transformed dataset.
from pathlib import Path
import json
import os
import time

EVAL_REPORT_DIR = REPORT_DIR / 'w1_aug_test_tta'
EVAL_REPORT_DIR.mkdir(parents=True, exist_ok=True)

if summary_df.empty:
    raise RuntimeError('W1 training produced no summary row.')

w1_row = summary_df.sort_values(
    ['healthy_aware_labeled_val_mask_map50', 'labeled_val_mask_map50'],
    ascending=False,
).iloc[0]
W1_CHECKPOINT = Path(w1_row['best_pt'])
W1_EXPERIMENT_KEY = str(w1_row['experiment'])
W1_DATASET_DIR = EXPERIMENT_ROOT / W1_EXPERIMENT_KEY / 'dataset'
W1_DATA_YAML = W1_DATASET_DIR / 'data.yaml'

if not W1_CHECKPOINT.exists():
    raise FileNotFoundError(f'Missing newly trained W1 checkpoint: {W1_CHECKPOINT}')
if not W1_DATASET_DIR.exists():
    raise FileNotFoundError(f'Missing Fourier-transformed W1 dataset: {W1_DATASET_DIR}')

actual_fingerprint = str(w1_row.get('split_fingerprint', ''))
if not SMOKE_RUN and actual_fingerprint != EXPECTED_SPLIT_FINGERPRINT:
    raise RuntimeError(
        'Split fingerprint mismatch. Refusing post-training comparison. '
        f'Expected {EXPECTED_SPLIT_FINGERPRINT}, got {actual_fingerprint}'
    )

TEST_VIEWS = [
    'original',
    'horizontal_flip',
    'vertical_flip',
    'rotate_90',
    'rotate_180',
    'rotate_270',
]

EVALUATION_CONTRACT = {
    'checkpoint': str(W1_CHECKPOINT),
    'dataset': 'shrimpdishandsegv2-v1',
    'seed': 42,
    'split_policy': 'stratified_grouped_specimen',
    'split_fingerprint': actual_fingerprint,
    'ultralytics_version': ultralytics.__version__,
    'imgsz': TRAIN_IMGSZ,
    'w1_sigma': FOURIER_SIGMA,
    'w1_alpha': FOURIER_ALPHA,
    'standard_ap_conf': STANDARD_AP_CONF,
    'diagnostic_conf': DIAGNOSTIC_CONF,
    'nms_iou': 0.70,
    'max_det': 300,
    'test_views': TEST_VIEWS,
    'processing_order': 'geometric_view_then_w1_then_yolo',
}
(EVAL_REPORT_DIR / '048_evaluation_contract.json').write_text(
    json.dumps(EVALUATION_CONTRACT, indent=2), encoding='utf-8'
)

print('W1 checkpoint:', W1_CHECKPOINT)
print('W1 transformed dataset:', W1_DATASET_DIR)
print('Verified split fingerprint:', actual_fingerprint)
print('Evaluation modes:', {
    'M0': RUN_M0_STANDARD_REFERENCE,
    'M1': RUN_M1_OFFLINE_AUG_TEST,
    'M2': RUN_M2_SINGLE_VIEW_CONTROL,
    'M3': RUN_M3_SIX_VIEW_TTA,
})


## M0 and M1: Standard AP Evaluation


In [ ]:
# Standard Ultralytics evaluation and offline augmented-test helpers.
from collections import defaultdict
import gc
import os
import shutil


def list_images_flat(image_dir):
    paths = []
    for ext in IMAGE_EXTENSIONS:
        paths.extend(Path(image_dir).glob(f'*{ext}'))
    return sorted(path for path in paths if path.is_file())


def label_for_image(image_path):
    return Path(image_path).parent.parent / 'labels' / f'{Path(image_path).stem}.txt'


def label_has_instances(label_path):
    label_path = Path(label_path)
    return label_path.exists() and any(line.strip() for line in label_path.read_text().splitlines())


def write_image_list_yaml(image_paths, key):
    list_dir = EVAL_REPORT_DIR / 'image_lists'
    list_dir.mkdir(parents=True, exist_ok=True)
    list_path = list_dir / f'{key}.txt'
    list_path.write_text('\n'.join(str(Path(path).resolve()) for path in image_paths) + '\n', encoding='utf-8')

    cfg = yaml.safe_load(W1_DATA_YAML.read_text(encoding='utf-8'))
    cfg['train'] = str(W1_DATASET_DIR / 'train' / 'images')
    cfg['val'] = str(W1_DATASET_DIR / 'valid' / 'images')
    cfg['test'] = str(list_path.resolve())
    yaml_path = list_dir / f'{key}.yaml'
    yaml_path.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding='utf-8')
    return yaml_path


def metric_or_nan(metrics, family, attribute):
    obj = getattr(metrics, family, None)
    if obj is None or not hasattr(obj, attribute):
        return float('nan')
    try:
        return float(getattr(obj, attribute))
    except Exception:
        return float('nan')


def per_class_rows(metrics, mode, view, scope, class_names):
    rows = []
    box_maps = list(getattr(metrics.box, 'maps', [])) if getattr(metrics, 'box', None) is not None else []
    mask_maps = list(getattr(metrics.seg, 'maps', [])) if getattr(metrics, 'seg', None) is not None else []
    for class_id, class_name in enumerate(class_names):
        rows.append({
            'mode': mode,
            'view': view,
            'scope': scope,
            'class_id': class_id,
            'class_name': class_name,
            'box_map50_95': float(box_maps[class_id]) if class_id < len(box_maps) else float('nan'),
            'mask_map50_95': float(mask_maps[class_id]) if class_id < len(mask_maps) else float('nan'),
        })
    return rows


def fixed_threshold_diagnostics(model, image_paths, conf=DIAGNOSTIC_CONF):
    image_paths = [Path(path) for path in image_paths]
    if not image_paths:
        return {}
    predictions = model.predict(
        source=[str(path) for path in image_paths],
        imgsz=TRAIN_IMGSZ,
        conf=conf,
        iou=0.70,
        max_det=300,
        batch=VAL_BATCH,
        stream=True,
        verbose=False,
    )
    healthy_images = healthy_with_mask_fp = healthy_masks = 0
    disease_images = disease_mask_misses = 0
    labeled_mask_errors = []
    labeled_mask_exact = []
    pred_masks_total = gt_total = 0

    for image_path, result in zip(image_paths, predictions):
        label_path = label_for_image(image_path)
        gt_count = len([line for line in label_path.read_text().splitlines() if line.strip()]) if label_path.exists() else 0
        pred_count = len(result.masks) if result.masks is not None else 0
        pred_masks_total += pred_count
        gt_total += gt_count
        if gt_count == 0:
            healthy_images += 1
            healthy_with_mask_fp += int(pred_count > 0)
            healthy_masks += pred_count
        else:
            disease_images += 1
            disease_mask_misses += int(pred_count == 0)
            labeled_mask_errors.append(abs(pred_count - gt_count) / max(1, gt_count))
            labeled_mask_exact.append(float(pred_count == gt_count))

    healthy_fp_rate = healthy_with_mask_fp / healthy_images if healthy_images else float('nan')
    healthy_fp_per_image = healthy_masks / healthy_images if healthy_images else float('nan')
    disease_miss_rate = disease_mask_misses / disease_images if disease_images else float('nan')
    mask_count_mae = float(np.mean(labeled_mask_errors)) if labeled_mask_errors else float('nan')
    mask_count_exact = float(np.mean(labeled_mask_exact)) if labeled_mask_exact else float('nan')
    return {
        'diagnostic_conf': conf,
        'healthy_images': healthy_images,
        'healthy_test_mask_fp_rate': healthy_fp_rate,
        'healthy_test_fp_masks_per_image': healthy_fp_per_image,
        'disease_images': disease_images,
        'labeled_test_disease_mask_miss_rate': disease_miss_rate,
        'labeled_test_mask_count_mae': mask_count_mae,
        'labeled_test_mask_count_exact': mask_count_exact,
        'gt_instances': gt_total,
        'pred_masks_at_diagnostic_conf': pred_masks_total,
    }


def standard_eval_row(model, mode, view, image_dir, class_names, fourier_seconds=float('nan')):
    image_paths = list_images_flat(image_dir)
    labeled_paths = [path for path in image_paths if label_has_instances(label_for_image(path))]
    if not image_paths or not labeled_paths:
        raise RuntimeError(f'Incomplete evaluation view {view}: all={len(image_paths)}, labeled={len(labeled_paths)}')

    full_yaml = write_image_list_yaml(image_paths, f'{mode}_{view}_full')
    labeled_yaml = write_image_list_yaml(labeled_paths, f'{mode}_{view}_labeled')

    full_metrics = model.val(
        data=str(full_yaml), split='test', imgsz=TRAIN_IMGSZ, batch=VAL_BATCH,
        conf=STANDARD_AP_CONF, iou=0.70, max_det=300, plots=False, verbose=False,
    )
    labeled_metrics = model.val(
        data=str(labeled_yaml), split='test', imgsz=TRAIN_IMGSZ, batch=VAL_BATCH,
        conf=STANDARD_AP_CONF, iou=0.70, max_det=300, plots=False, verbose=False,
    )
    diagnostics = fixed_threshold_diagnostics(model, image_paths)
    labeled_map50 = metric_or_nan(labeled_metrics, 'seg', 'map50')
    healthy_aware = (
        labeled_map50
        - COUNT_PENALTY_WEIGHT * diagnostics['labeled_test_mask_count_mae']
        - DISEASE_MISS_PENALTY_WEIGHT * diagnostics['labeled_test_disease_mask_miss_rate']
        - HEALTHY_FP_PENALTY_WEIGHT * diagnostics['healthy_test_mask_fp_rate']
    )
    row = {
        'mode': mode,
        'view': view,
        'evaluator': 'ultralytics_val',
        'ap_conf': STANDARD_AP_CONF,
        'images': len(image_paths),
        'labeled_images': len(labeled_paths),
        'healthy_images': len(image_paths) - len(labeled_paths),
        'fourier_precompute_seconds': fourier_seconds,
        'full_test_box_precision': metric_or_nan(full_metrics, 'box', 'mp'),
        'full_test_box_recall': metric_or_nan(full_metrics, 'box', 'mr'),
        'full_test_box_map50': metric_or_nan(full_metrics, 'box', 'map50'),
        'full_test_box_map50_95': metric_or_nan(full_metrics, 'box', 'map'),
        'full_test_mask_precision': metric_or_nan(full_metrics, 'seg', 'mp'),
        'full_test_mask_recall': metric_or_nan(full_metrics, 'seg', 'mr'),
        'full_test_mask_map50': metric_or_nan(full_metrics, 'seg', 'map50'),
        'full_test_mask_map50_95': metric_or_nan(full_metrics, 'seg', 'map'),
        'labeled_test_box_map50': metric_or_nan(labeled_metrics, 'box', 'map50'),
        'labeled_test_box_map50_95': metric_or_nan(labeled_metrics, 'box', 'map'),
        'labeled_test_mask_map50': labeled_map50,
        'labeled_test_mask_map50_95': metric_or_nan(labeled_metrics, 'seg', 'map'),
        **diagnostics,
        'healthy_aware_labeled_test_mask_map50': healthy_aware,
    }
    class_rows = (
        per_class_rows(full_metrics, mode, view, 'full', class_names)
        + per_class_rows(labeled_metrics, mode, view, 'labeled', class_names)
    )
    del full_metrics, labeled_metrics
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return row, class_rows


def apply_test_view(image, view):
    if view == 'original':
        return image.copy()
    if view == 'horizontal_flip':
        return cv2.flip(image, 1)
    if view == 'vertical_flip':
        return cv2.flip(image, 0)
    if view == 'rotate_90':
        return cv2.rotate(image, cv2.ROTATE_90_CLOCKWISE)
    if view == 'rotate_180':
        return cv2.rotate(image, cv2.ROTATE_180)
    if view == 'rotate_270':
        return cv2.rotate(image, cv2.ROTATE_90_COUNTERCLOCKWISE)
    raise ValueError(view)


def transform_normalized_polygon(points, view):
    points = np.asarray(points, dtype=float).copy()
    x, y = points[:, 0].copy(), points[:, 1].copy()
    if view == 'original':
        pass
    elif view == 'horizontal_flip':
        points[:, 0], points[:, 1] = 1.0 - x, y
    elif view == 'vertical_flip':
        points[:, 0], points[:, 1] = x, 1.0 - y
    elif view == 'rotate_90':
        points[:, 0], points[:, 1] = 1.0 - y, x
    elif view == 'rotate_180':
        points[:, 0], points[:, 1] = 1.0 - x, 1.0 - y
    elif view == 'rotate_270':
        points[:, 0], points[:, 1] = y, 1.0 - x
    else:
        raise ValueError(view)
    return points.clip(0.0, 1.0)


def transform_segmentation_label(source_label, destination_label, view):
    source_label = Path(source_label)
    destination_label = Path(destination_label)
    output_lines = []
    if source_label.exists():
        for line in source_label.read_text().splitlines():
            parts = line.strip().split()
            if not parts:
                continue
            if len(parts) < 7 or (len(parts) - 1) % 2:
                raise ValueError(f'Expected YOLO polygon label, got: {source_label} :: {line}')
            class_id = parts[0]
            points = np.asarray([float(value) for value in parts[1:]], dtype=float).reshape(-1, 2)
            transformed = transform_normalized_polygon(points, view)
            coords = ' '.join(f'{value:.8f}' for value in transformed.reshape(-1))
            output_lines.append(f'{class_id} {coords}')
    destination_label.write_text('\n'.join(output_lines) + ('\n' if output_lines else ''), encoding='utf-8')


def hardlink_or_copy(source, destination):
    destination.parent.mkdir(parents=True, exist_ok=True)
    try:
        os.link(source, destination)
    except Exception:
        shutil.copy2(source, destination)


def build_w1_test_views():
    root = EXPERIMENT_ROOT / 'offline_test_views_w1'
    if root.exists():
        shutil.rmtree(root)
    source_images = list_images_flat(Path(base_path) / 'test' / 'images')
    source_labels = Path(base_path) / 'test' / 'labels'
    canonical_w1_images = W1_DATASET_DIR / 'test' / 'images'
    timings = {}

    for view in TEST_VIEWS:
        view_images = root / view / 'images'
        view_labels = root / view / 'labels'
        view_images.mkdir(parents=True, exist_ok=True)
        view_labels.mkdir(parents=True, exist_ok=True)

        if view == 'original':
            started = time.perf_counter()
            for image_path in source_images:
                canonical_image = canonical_w1_images / image_path.name
                if not canonical_image.exists():
                    raise FileNotFoundError(canonical_image)
                shutil.copy2(canonical_image, view_images / image_path.name)
                shutil.copy2(source_labels / f'{image_path.stem}.txt', view_labels / f'{image_path.stem}.txt')
            timings[view] = float('nan')  # Canonical original-view W1 was created during training.
            continue

        started = time.perf_counter()
        image_batch, path_batch, batch_shape = [], [], None

        def flush_fourier_batch():
            nonlocal image_batch, path_batch, batch_shape
            if image_batch:
                apply_fourier_batch(
                    image_batch, path_batch, sigma=FOURIER_SIGMA,
                    alpha=FOURIER_ALPHA, device=fourier_device()
                )
            image_batch, path_batch, batch_shape = [], [], None

        for image_path in source_images:
            image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
            if image is None:
                raise FileNotFoundError(image_path)
            transformed = apply_test_view(image, view)
            output_name = f'{view}__{image_path.name}'
            output_image = view_images / output_name
            transform_segmentation_label(
                source_labels / f'{image_path.stem}.txt',
                view_labels / f'{Path(output_name).stem}.txt',
                view,
            )
            if batch_shape is not None and transformed.shape != batch_shape:
                flush_fourier_batch()
            image_batch.append(transformed)
            path_batch.append(output_image)
            batch_shape = transformed.shape
            if len(image_batch) >= FOURIER_BATCH_SIZE:
                flush_fourier_batch()
        flush_fourier_batch()
        timings[view] = time.perf_counter() - started

    pooled_images = root / 'pooled' / 'images'
    pooled_labels = root / 'pooled' / 'labels'
    pooled_images.mkdir(parents=True, exist_ok=True)
    pooled_labels.mkdir(parents=True, exist_ok=True)
    for view in TEST_VIEWS:
        for image_path in list_images_flat(root / view / 'images'):
            output_name = image_path.name if view != 'original' else f'original__{image_path.name}'
            hardlink_or_copy(image_path, pooled_images / output_name)
            hardlink_or_copy(label_for_image(image_path), pooled_labels / f'{Path(output_name).stem}.txt')

    expected = len(source_images) * len(TEST_VIEWS)
    actual = len(list_images_flat(pooled_images))
    if actual != expected:
        raise RuntimeError(f'Expected {expected} pooled images, found {actual}')
    (EVAL_REPORT_DIR / '048_offline_view_fourier_timing.json').write_text(
        json.dumps(timings, indent=2), encoding='utf-8'
    )
    return root, timings


In [ ]:
# Run M0 and M1 with standard Ultralytics validation AP.
data_cfg = yaml.safe_load(W1_DATA_YAML.read_text(encoding='utf-8'))
names_obj = data_cfg.get('names', {})
if isinstance(names_obj, dict):
    CLASS_NAMES = [names_obj[key] for key in sorted(names_obj, key=lambda value: int(value))]
else:
    CLASS_NAMES = list(names_obj)

w1_model = YOLO(str(W1_CHECKPOINT))
standard_rows = []
standard_class_rows = []

if RUN_M0_STANDARD_REFERENCE:
    row, class_rows = standard_eval_row(
        w1_model, 'M0_standard_w1_reference', 'original', W1_DATASET_DIR / 'test' / 'images', CLASS_NAMES
    )
    standard_rows.append(row)
    standard_class_rows.extend(class_rows)
    print('M0 complete')
    display(pd.DataFrame([row]))

OFFLINE_VIEW_ROOT = None
offline_fourier_timings = {}
if RUN_M1_OFFLINE_AUG_TEST:
    OFFLINE_VIEW_ROOT, offline_fourier_timings = build_w1_test_views()

if RUN_M1_OFFLINE_AUG_TEST:
    for view in TEST_VIEWS:
        row, class_rows = standard_eval_row(
            w1_model,
            'M1_offline_augmented_test',
            view,
            OFFLINE_VIEW_ROOT / view / 'images',
            CLASS_NAMES,
            fourier_seconds=offline_fourier_timings.get(view, float('nan')),
        )
        standard_rows.append(row)
        standard_class_rows.extend(class_rows)
        print(f'M1 view complete: {view}')

    row, class_rows = standard_eval_row(
        w1_model,
        'M1_offline_augmented_test',
        'pooled_six_views',
        OFFLINE_VIEW_ROOT / 'pooled' / 'images',
        CLASS_NAMES,
        fourier_seconds=sum(offline_fourier_timings.values()),
    )
    standard_rows.append(row)
    standard_class_rows.extend(class_rows)

standard_metrics_df = pd.DataFrame(standard_rows)
standard_per_class_df = pd.DataFrame(standard_class_rows)
standard_metrics_csv = EVAL_REPORT_DIR / '048_w1_standard_and_aug_test_metrics.csv'
standard_per_class_csv = EVAL_REPORT_DIR / '048_w1_standard_and_aug_test_per_class.csv'
standard_metrics_df.to_csv(standard_metrics_csv, index=False)
standard_per_class_df.to_csv(standard_per_class_csv, index=False)
print('Saved:', standard_metrics_csv)
print('Saved:', standard_per_class_csv)
display(standard_metrics_df)


## M2 and M3: Matched Custom TTA Evaluation


In [ ]:
# Memory-conscious custom evaluator and W1 TTA helpers.
from collections import defaultdict
import pickle

TTA_CONFIG = {
    'transforms': TEST_VIEWS,
    'candidate_conf': STANDARD_AP_CONF,
    'diagnostic_conf': DIAGNOSTIC_CONF,
    'iou_nms': 0.70,
    'max_det': 300,
    'matching_metric': 'mask_iou',
    'matching_iou_threshold': 0.50,
    'matching_assignment': 'greedy_best_match',
    'one_to_one': True,
    'class_aware_matching': True,
    'minimum_support': 4,
    'total_views': 6,
    'mask_fusion': 'average_binary_masks',
    'mask_threshold': 0.50,
    'final_mask_nms_iou': 0.50,
}
(EVAL_REPORT_DIR / '048_tta_config.json').write_text(json.dumps(TTA_CONFIG, indent=2), encoding='utf-8')


def inverse_mask(mask, view, original_hw):
    mask = np.ascontiguousarray(mask, dtype=np.uint8)
    if view == 'horizontal_flip':
        mask = cv2.flip(mask, 1)
    elif view == 'vertical_flip':
        mask = cv2.flip(mask, 0)
    elif view == 'rotate_90':
        mask = cv2.rotate(mask, cv2.ROTATE_90_COUNTERCLOCKWISE)
    elif view == 'rotate_180':
        mask = cv2.rotate(mask, cv2.ROTATE_180)
    elif view == 'rotate_270':
        mask = cv2.rotate(mask, cv2.ROTATE_90_CLOCKWISE)
    elif view != 'original':
        raise ValueError(view)
    height, width = original_hw
    mask = cv2.resize(mask, (width, height), interpolation=cv2.INTER_NEAREST)
    return mask > 0


def boolean_mask_iou(mask_a, mask_b):
    intersection = np.logical_and(mask_a, mask_b).sum()
    union = np.logical_or(mask_a, mask_b).sum()
    return float(intersection / union) if union else 0.0


def mask_box(mask):
    ys, xs = np.where(mask)
    if not len(xs):
        return np.asarray([0.0, 0.0, 0.0, 0.0], dtype=float)
    return np.asarray([xs.min(), ys.min(), xs.max() + 1, ys.max() + 1], dtype=float)


def pack_instance(instance):
    mask = np.asarray(instance['mask'], dtype=bool)
    return {
        'class_id': int(instance['class_id']),
        'confidence': float(instance.get('confidence', 1.0)),
        'support': int(instance.get('support', 1)),
        'shape': tuple(mask.shape),
        'packed_mask': np.packbits(mask.reshape(-1)),
        'box': mask_box(mask),
    }


POPCOUNT = np.asarray([int(value).bit_count() for value in range(256)], dtype=np.uint8)


def packed_mask_iou(instance_a, instance_b):
    if tuple(instance_a['shape']) != tuple(instance_b['shape']):
        raise ValueError('Packed masks must have equal shapes')
    left = instance_a['packed_mask']
    right = instance_b['packed_mask']
    intersection = POPCOUNT[np.bitwise_and(left, right)].sum(dtype=np.uint64)
    union = POPCOUNT[np.bitwise_or(left, right)].sum(dtype=np.uint64)
    return float(intersection / union) if union else 0.0


def box_iou_xyxy(box_a, box_b):
    x1, y1 = max(box_a[0], box_b[0]), max(box_a[1], box_b[1])
    x2, y2 = min(box_a[2], box_b[2]), min(box_a[3], box_b[3])
    intersection = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    area_a = max(0.0, box_a[2] - box_a[0]) * max(0.0, box_a[3] - box_a[1])
    area_b = max(0.0, box_b[2] - box_b[0]) * max(0.0, box_b[3] - box_b[1])
    union = area_a + area_b - intersection
    return float(intersection / union) if union else 0.0


def predict_tta_view(model, image, view, original_hw):
    transformed = apply_test_view(image, view)
    fourier_started = time.perf_counter()
    transformed = fourier_highpass_boost_image_cpu(
        transformed, sigma=FOURIER_SIGMA, alpha=FOURIER_ALPHA
    )
    fourier_seconds = time.perf_counter() - fourier_started

    inference_started = time.perf_counter()
    result = model.predict(
        source=transformed,
        imgsz=TRAIN_IMGSZ,
        conf=TTA_CONFIG['candidate_conf'],
        iou=TTA_CONFIG['iou_nms'],
        max_det=TTA_CONFIG['max_det'],
        agnostic_nms=False,
        verbose=False,
    )[0]
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    inference_seconds = time.perf_counter() - inference_started

    instances = []
    if result.masks is not None and result.boxes is not None:
        raw_masks = result.masks.data.detach().cpu().numpy()
        classes = result.boxes.cls.detach().cpu().numpy().astype(int)
        scores = result.boxes.conf.detach().cpu().numpy()
        view_height, view_width = transformed.shape[:2]
        for raw_mask, class_id, score in zip(raw_masks, classes, scores):
            view_mask = cv2.resize(
                raw_mask, (view_width, view_height), interpolation=cv2.INTER_NEAREST
            ) >= 0.5
            instances.append({
                'mask': inverse_mask(view_mask, view, original_hw),
                'class_id': int(class_id),
                'confidence': float(score),
                'view': view,
            })
    del result, transformed
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return instances, fourier_seconds, inference_seconds


def fused_track_mask(track, threshold):
    accumulator = np.zeros_like(track[0]['mask'], dtype=np.uint16)
    for instance in track:
        accumulator += instance['mask'].astype(np.uint16)
    return (accumulator / len(track)) >= threshold


def fuse_tta_views(view_predictions, transforms, minimum_support):
    tracks = []
    for view in transforms:
        instances = view_predictions.get(view, [])
        candidates = []
        for track_id, track in enumerate(tracks):
            reference = fused_track_mask(track, TTA_CONFIG['mask_threshold'])
            for instance_id, instance in enumerate(instances):
                if TTA_CONFIG['class_aware_matching'] and track[0]['class_id'] != instance['class_id']:
                    continue
                overlap = boolean_mask_iou(reference, instance['mask'])
                if overlap >= TTA_CONFIG['matching_iou_threshold']:
                    candidates.append((overlap, track_id, instance_id))

        used_tracks, used_instances = set(), set()
        for _, track_id, instance_id in sorted(candidates, reverse=True):
            if track_id in used_tracks or instance_id in used_instances:
                continue
            tracks[track_id].append(instances[instance_id])
            used_tracks.add(track_id)
            used_instances.add(instance_id)
        tracks.extend([[instance] for instance_id, instance in enumerate(instances) if instance_id not in used_instances])

    fused = []
    for track in tracks:
        if len(track) < minimum_support:
            continue
        fused_mask = fused_track_mask(track, TTA_CONFIG['mask_threshold'])
        if not fused_mask.any():
            continue
        fused.append({
            'mask': fused_mask,
            'class_id': track[0]['class_id'],
            'confidence': float(np.mean([instance['confidence'] for instance in track])),
            'support': len(track),
        })

    kept = []
    for prediction in sorted(fused, key=lambda item: item['confidence'], reverse=True):
        suppress = any(
            prediction['class_id'] == old['class_id']
            and boolean_mask_iou(prediction['mask'], old['mask']) >= TTA_CONFIG['final_mask_nms_iou']
            for old in kept
        )
        if not suppress:
            kept.append(prediction)
    return kept


def load_gt_instances(label_path, image_hw):
    height, width = image_hw
    instances = []
    label_path = Path(label_path)
    if not label_path.exists():
        return instances
    for line in label_path.read_text().splitlines():
        parts = line.strip().split()
        if len(parts) < 7:
            continue
        class_id = int(float(parts[0]))
        points = np.asarray(parts[1:], dtype=float).reshape(-1, 2)
        points[:, 0] *= width
        points[:, 1] *= height
        mask = np.zeros((height, width), dtype=np.uint8)
        cv2.fillPoly(mask, [np.round(points).astype(np.int32)], 1)
        instances.append(pack_instance({'class_id': class_id, 'confidence': 1.0, 'mask': mask > 0}))
    return instances


def ap_from_pr(recall, precision):
    mrec = np.r_[0.0, recall, 1.0]
    mpre = np.r_[1.0, precision, 0.0]
    mpre = np.maximum.accumulate(mpre[::-1])[::-1]
    grid = np.linspace(0.0, 1.0, 101)
    values = np.interp(grid, mrec, mpre)
    return float(np.trapezoid(values, grid))


def evaluate_one_threshold(records, class_id, iou_threshold, metric_type):
    total_gt = sum(sum(gt['class_id'] == class_id for gt in record['gt']) for record in records)
    ranked = []
    for image_id, record in enumerate(records):
        ranked.extend(
            (prediction['confidence'], image_id, prediction)
            for prediction in record['pred'] if prediction['class_id'] == class_id
        )
    ranked.sort(key=lambda item: item[0], reverse=True)
    matched = defaultdict(set)
    true_positive, false_positive = [], []
    for _, image_id, prediction in ranked:
        candidates = []
        for gt_id, gt in enumerate(records[image_id]['gt']):
            if gt['class_id'] != class_id or gt_id in matched[image_id]:
                continue
            overlap = (
                packed_mask_iou(prediction, gt)
                if metric_type == 'mask'
                else box_iou_xyxy(prediction['box'], gt['box'])
            )
            candidates.append((overlap, gt_id))
        best_iou, best_gt = max(candidates, default=(0.0, -1))
        hit = best_iou >= iou_threshold
        true_positive.append(float(hit))
        false_positive.append(float(not hit))
        if hit:
            matched[image_id].add(best_gt)

    tp_sum = float(sum(true_positive))
    fp_sum = float(sum(false_positive))
    precision = tp_sum / max(tp_sum + fp_sum, 1e-12)
    recall = tp_sum / max(total_gt, 1e-12)
    if total_gt and ranked:
        tp_curve = np.cumsum(true_positive)
        fp_curve = np.cumsum(false_positive)
        recall_curve = tp_curve / total_gt
        precision_curve = tp_curve / np.maximum(tp_curve + fp_curve, 1e-12)
        average_precision = ap_from_pr(recall_curve, precision_curve)
    else:
        average_precision = 0.0 if total_gt else float('nan')
    return {
        'precision': precision,
        'recall': recall,
        'ap': average_precision,
        'gt': total_gt,
        'pred': len(ranked),
    }


def evaluate_custom_ap(records, class_names, mode, scope):
    thresholds = np.arange(0.50, 0.96, 0.05)
    per_class = []
    for class_id, class_name in enumerate(class_names):
        row = {'mode': mode, 'scope': scope, 'class_id': class_id, 'class_name': class_name}
        for metric_type, suffix in (('box', 'box'), ('mask', 'mask')):
            evaluations = [
                evaluate_one_threshold(records, class_id, float(threshold), metric_type)
                for threshold in thresholds
            ]
            row.update({
                f'{suffix}_precision': evaluations[0]['precision'],
                f'{suffix}_recall': evaluations[0]['recall'],
                f'{suffix}_map50': evaluations[0]['ap'],
                f'{suffix}_map50_95': float(np.nanmean([item['ap'] for item in evaluations])),
            })
            if metric_type == 'mask':
                row['gt_instances'] = evaluations[0]['gt']
                row['pred_instances'] = evaluations[0]['pred']
        per_class.append(row)

    frame = pd.DataFrame(per_class)
    valid = frame[frame['gt_instances'] > 0]
    overall = {
        'mode': mode,
        'scope': scope,
        'evaluator': 'custom_tta_101_point_ap',
        'ap_conf': TTA_CONFIG['candidate_conf'],
        'box_precision': float(valid['box_precision'].mean()),
        'box_recall': float(valid['box_recall'].mean()),
        'box_map50': float(valid['box_map50'].mean()),
        'box_map50_95': float(valid['box_map50_95'].mean()),
        'mask_precision': float(valid['mask_precision'].mean()),
        'mask_recall': float(valid['mask_recall'].mean()),
        'mask_map50': float(valid['mask_map50'].mean()),
        'mask_map50_95': float(valid['mask_map50_95'].mean()),
    }
    return overall, per_class


def custom_diagnostics(records, threshold=DIAGNOSTIC_CONF):
    healthy_images = healthy_with_fp = healthy_masks = 0
    disease_images = disease_misses = 0
    count_errors = []
    count_exact = []
    for record in records:
        predictions = [prediction for prediction in record['pred'] if prediction['confidence'] >= threshold]
        gt_count = len(record['gt'])
        pred_count = len(predictions)
        if gt_count == 0:
            healthy_images += 1
            healthy_with_fp += int(pred_count > 0)
            healthy_masks += pred_count
        else:
            disease_images += 1
            disease_misses += int(pred_count == 0)
            count_errors.append(abs(pred_count - gt_count) / max(1, gt_count))
            count_exact.append(float(pred_count == gt_count))
    return {
        'diagnostic_conf': threshold,
        'healthy_images': healthy_images,
        'healthy_test_mask_fp_rate': healthy_with_fp / healthy_images if healthy_images else float('nan'),
        'healthy_test_fp_masks_per_image': healthy_masks / healthy_images if healthy_images else float('nan'),
        'disease_images': disease_images,
        'labeled_test_disease_mask_miss_rate': disease_misses / disease_images if disease_images else float('nan'),
        'labeled_test_mask_count_mae': float(np.mean(count_errors)) if count_errors else float('nan'),
        'labeled_test_mask_count_exact': float(np.mean(count_exact)) if count_exact else float('nan'),
    }


In [ ]:
# Run M2 and M3 in one pass over the original test images.
tta_rows = []
tta_per_class_rows = []
tta_timing_rows = []

if RUN_M2_SINGLE_VIEW_CONTROL or RUN_M3_SIX_VIEW_TTA:
    tta_model = YOLO(str(W1_CHECKPOINT))
    source_test_images = list_images_flat(Path(base_path) / 'test' / 'images')
    source_test_labels = Path(base_path) / 'test' / 'labels'
    records_m2, records_m3 = [], []

    for image_index, image_path in enumerate(source_test_images, 1):
        image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
        if image is None:
            raise FileNotFoundError(image_path)

        view_predictions = {}
        fourier_seconds = inference_seconds = 0.0
        original_fourier_seconds = original_inference_seconds = 0.0
        for view in TEST_VIEWS:
            instances, view_fourier_seconds, view_inference_seconds = predict_tta_view(
                tta_model, image, view, image.shape[:2]
            )
            view_predictions[view] = instances
            fourier_seconds += view_fourier_seconds
            inference_seconds += view_inference_seconds
            if view == 'original':
                original_fourier_seconds = view_fourier_seconds
                original_inference_seconds = view_inference_seconds

        gt = load_gt_instances(source_test_labels / f'{image_path.stem}.txt', image.shape[:2])

        if RUN_M2_SINGLE_VIEW_CONTROL:
            fusion_started = time.perf_counter()
            single_predictions = fuse_tta_views(view_predictions, ['original'], minimum_support=1)
            single_fusion_seconds = time.perf_counter() - fusion_started
            records_m2.append({
                'path': str(image_path),
                'gt': gt,
                'pred': [pack_instance(prediction) for prediction in single_predictions],
            })
            tta_timing_rows.append({
                'mode': 'M2_single_view_custom_control',
                'image': image_path.name,
                'fourier_seconds': original_fourier_seconds,
                'inference_seconds': original_inference_seconds,
                'fusion_seconds': single_fusion_seconds,
                'total_seconds': original_fourier_seconds + original_inference_seconds + single_fusion_seconds,
            })

        if RUN_M3_SIX_VIEW_TTA:
            fusion_started = time.perf_counter()
            fused_predictions = fuse_tta_views(
                view_predictions, TEST_VIEWS, minimum_support=TTA_CONFIG['minimum_support']
            )
            fusion_seconds = time.perf_counter() - fusion_started
            records_m3.append({
                'path': str(image_path),
                'gt': gt,
                'pred': [pack_instance(prediction) for prediction in fused_predictions],
            })
            tta_timing_rows.append({
                'mode': 'M3_six_view_w1_tta',
                'image': image_path.name,
                'fourier_seconds': fourier_seconds,
                'inference_seconds': inference_seconds,
                'fusion_seconds': fusion_seconds,
                'total_seconds': fourier_seconds + inference_seconds + fusion_seconds,
            })

        del view_predictions, image
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        if image_index % 10 == 0 or image_index == len(source_test_images):
            print(f'TTA inference: {image_index}/{len(source_test_images)}')
            with open(EVAL_REPORT_DIR / '048_tta_partial_records.pkl', 'wb') as stream:
                pickle.dump({'M2': records_m2, 'M3': records_m3}, stream, protocol=pickle.HIGHEST_PROTOCOL)

    for mode, records in (
        ('M2_single_view_custom_control', records_m2),
        ('M3_six_view_w1_tta', records_m3),
    ):
        if not records:
            continue
        labeled_records = [record for record in records if record['gt']]
        full_metrics, full_per_class = evaluate_custom_ap(records, CLASS_NAMES, mode, 'full')
        labeled_metrics, labeled_per_class = evaluate_custom_ap(labeled_records, CLASS_NAMES, mode, 'labeled')
        diagnostics = custom_diagnostics(records)
        healthy_aware = (
            labeled_metrics['mask_map50']
            - COUNT_PENALTY_WEIGHT * diagnostics['labeled_test_mask_count_mae']
            - DISEASE_MISS_PENALTY_WEIGHT * diagnostics['labeled_test_disease_mask_miss_rate']
            - HEALTHY_FP_PENALTY_WEIGHT * diagnostics['healthy_test_mask_fp_rate']
        )
        timing_frame = pd.DataFrame([row for row in tta_timing_rows if row['mode'] == mode])
        row = {
            'mode': mode,
            'evaluator': 'custom_tta_101_point_ap',
            'ap_conf': STANDARD_AP_CONF,
            'images': len(records),
            'labeled_images': len(labeled_records),
            'full_test_box_precision': full_metrics['box_precision'],
            'full_test_box_recall': full_metrics['box_recall'],
            'full_test_box_map50': full_metrics['box_map50'],
            'full_test_box_map50_95': full_metrics['box_map50_95'],
            'full_test_mask_precision': full_metrics['mask_precision'],
            'full_test_mask_recall': full_metrics['mask_recall'],
            'full_test_mask_map50': full_metrics['mask_map50'],
            'full_test_mask_map50_95': full_metrics['mask_map50_95'],
            'labeled_test_box_map50': labeled_metrics['box_map50'],
            'labeled_test_box_map50_95': labeled_metrics['box_map50_95'],
            'labeled_test_mask_map50': labeled_metrics['mask_map50'],
            'labeled_test_mask_map50_95': labeled_metrics['mask_map50_95'],
            **diagnostics,
            'healthy_aware_labeled_test_mask_map50': healthy_aware,
            'mean_fourier_ms_per_original': 1000.0 * timing_frame['fourier_seconds'].mean(),
            'mean_inference_ms_per_original': 1000.0 * timing_frame['inference_seconds'].mean(),
            'mean_fusion_ms_per_original': 1000.0 * timing_frame['fusion_seconds'].mean(),
            'mean_total_ms_per_original': 1000.0 * timing_frame['total_seconds'].mean(),
        }
        tta_rows.append(row)
        tta_per_class_rows.extend(full_per_class + labeled_per_class)

tta_metrics_df = pd.DataFrame(tta_rows)
tta_per_class_df = pd.DataFrame(tta_per_class_rows)
tta_timing_df = pd.DataFrame(tta_timing_rows)
tta_metrics_csv = EVAL_REPORT_DIR / '048_w1_tta_metrics.csv'
tta_per_class_csv = EVAL_REPORT_DIR / '048_w1_tta_per_class.csv'
tta_timing_csv = EVAL_REPORT_DIR / '048_w1_tta_timing.csv'
tta_metrics_df.to_csv(tta_metrics_csv, index=False)
tta_per_class_df.to_csv(tta_per_class_csv, index=False)
tta_timing_df.to_csv(tta_timing_csv, index=False)
print('Saved:', tta_metrics_csv)
print('Saved:', tta_per_class_csv)
print('Saved:', tta_timing_csv)
display(tta_metrics_df)


## Consolidated Evaluation Table


In [ ]:
# Consolidate all mode rows without implying identical evaluator implementations.
combined_frames = []
if 'standard_metrics_df' in globals() and not standard_metrics_df.empty:
    combined_frames.append(standard_metrics_df)
if 'tta_metrics_df' in globals() and not tta_metrics_df.empty:
    combined_frames.append(tta_metrics_df)

combined_metrics_df = pd.concat(combined_frames, ignore_index=True, sort=False) if combined_frames else pd.DataFrame()
combined_metrics_csv = EVAL_REPORT_DIR / '048_w1_all_evaluation_modes.csv'
combined_metrics_df.to_csv(combined_metrics_csv, index=False)

comparison_notes = {
    'M0_vs_M1': 'Valid standard Ultralytics AP comparison; M1 views are correlated copies, so per-view rows are primary.',
    'M2_vs_M3': 'Valid matched custom-evaluator comparison for the effect of six-view TTA.',
    'M0_vs_M2': 'Evaluator calibration only; custom mask rasterization and AP integration can create a delta.',
    'M0_vs_M3': 'Descriptive only unless the M0-to-M2 evaluator delta is disclosed.',
}
(EVAL_REPORT_DIR / '048_comparison_boundaries.json').write_text(
    json.dumps(comparison_notes, indent=2), encoding='utf-8'
)
print('Saved:', combined_metrics_csv)
display(combined_metrics_df)


## Final Download Checklist


In [ ]:
# Final download checklist: only references variables created unconditionally above.
import zipfile

training_summary_csv = REPORT_DIR / '048_w1_train_summary.csv'
training_paper_csv = REPORT_DIR / '048_w1_train_paper_row.csv'
training_partial_csv = REPORT_DIR / '048_w1_train_partial.csv'
train_args_path = REPORT_DIR / 'train_args_048_w1_train_aug_test_tta_seed42.json'

required_paths = [
    training_summary_csv,
    training_paper_csv,
    training_partial_csv,
    train_args_path,
    REPORT_DIR / 'split_manifests',
    W1_CHECKPOINT,
    EVAL_REPORT_DIR,
]

print('Final download checklist')
for index, path in enumerate(required_paths, 1):
    print(f'{index}. {path} | exists={Path(path).exists()}')

package_path = WORK_DIR / '048_w1_train_aug_test_tta_outputs.zip'
with zipfile.ZipFile(package_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for report_path in REPORT_DIR.rglob('*'):
        if report_path.is_file() and report_path.suffix.lower() != '.pkl':
            archive.write(report_path, arcname=str(Path('reports') / report_path.relative_to(REPORT_DIR)))
    archive.write(W1_CHECKPOINT, arcname='weights/w1_best.pt')
    results_csv = Path(w1_row['run_path']) / 'results.csv'
    if results_csv.exists():
        archive.write(results_csv, arcname='training/results.csv')

print('Compact reports-and-checkpoint package:', package_path)
print('Package size MB:', round(package_path.stat().st_size / (1024 * 1024), 2))
print('On Kaggle, download this ZIP from /kaggle/working after the run completes.')
